[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/14_vector_calculus_field_theorems/exercises.ipynb)

# Module 14 — Vector Calculus and Field Theorems: Exercises

Forty fully solved problems across four tiers. Every numeric answer below is recomputed by a
code cell that runs in this notebook; nothing is quoted from memory.

---

## How the tiers are organised

This package contains **40 fully solved problems** in four tiers:

- **L0 — Concept Checks (8 problems, `L0.1`–`L0.8`)**: orientation and sign conventions, the meaning of divergence and curl, and the punctured-domain counterexample.
- **L1 — Foundations (10 problems, `L1.1`–`L1.10`)**: line, surface and flux integrals from an explicit parameterisation, and each field theorem computed on both sides.
- **L2 — Applications (12 problems, `L2.1`–`L2.12`)**: electromagnetism, fluid mechanics, heat conduction, Neural ODEs, GAN update fields, PINN losses, Fokker–Planck.
- **L3 — Challenge Proofs (10 problems, `L3.1`–`L3.10`)**: higher-dimensional flux, pullbacks, winding-number monodromy, the monopole obstruction, and a stability synthesis.

The setup cell below is run once; every later code cell depends on it.

---

In [1]:
# Setup for every code cell below. Run this first.
import numpy as np
import sympy as sp
from scipy import integrate

rng = np.random.default_rng(0)
np.set_printoptions(precision=6, suppress=True)

def rel(a, b):
    """Relative error, safe at zero."""
    return abs(a - b) / max(1.0, abs(b))

print("numpy", np.__version__, "| sympy", sp.__version__)

numpy 2.4.6 | sympy 1.14.0


## L0 — Concept Checks

### Problem L0.1 — Scalar vs Vector Line Integrals Intuition
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 6  

**Problem Statement:**  
Explain the geometric and physical differences between the scalar line integral $\int_C f\,\mathrm{d}s$ and the vector line integral $\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}$. Specifically, how does reversing the orientation of curve $C$ to $-C$ affect each integral?

#### First-Principles Intuition
A scalar line integral accumulates a density function $f$ over curve segments of non-negative length $\mathrm{d}s = \lVert \mathbf{r}'(t) \rVert\mathrm{d}t$. It measures quantities like total mass or wire length, which do not depend on the direction of traversal. A vector line integral measures the projection of a vector field $\mathbf{F}$ along the directional unit tangent $\mathbf{T}$, accumulating directed work or circulation.

#### Step-by-Step Solution
1. Parameterize $-C$ by $\mathbf{g}(t) = \mathbf{r}(a + b - t)$ for $t \in [a, b]$.
2. For the scalar line integral:

$$
   \mathrm{d}s_{-C} = \lVert \mathbf{g}'(t) \rVert\,\mathrm{d}t = \lVert -\mathbf{r}'(a + b - t) \rVert\,\mathrm{d}t = \lVert \mathbf{r}'(u) \rVert\,\mathrm{d}u = \mathrm{d}s_C
$$

   Therefore:

$$
   \int_{-C} f\,\mathrm{d}s = \int_C f\,\mathrm{d}s
$$

3. For the vector line integral:

$$
   \mathrm{d}\mathbf{g}(t) = \mathbf{g}'(t)\,\mathrm{d}t = -\mathbf{r}'(u)\,\mathrm{d}u = -\mathrm{d}\mathbf{r}
$$

   Therefore:

$$
   \int_{-C} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = -\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}
$$

$$
\boxed{\int_{-C} f\,\mathrm{d}s = \int_C f\,\mathrm{d}s \quad \text{and} \quad \int_{-C} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = -\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}}
$$

#### Key Insight / Takeaway
Scalar line integrals are orientation-invariant (measuring unoriented geometric quantity), whereas vector line integrals are orientation-reversing (measuring oriented projection along motion).

---

### Problem L0.2 — Physical Interpretation of Divergence
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 2  

**Problem Statement:**  
Consider the 2D vector field $\mathbf{F}_1(x, y) = x\mathbf{i} + y\mathbf{j}$ and $\mathbf{F}_2(x, y) = -y\mathbf{i} + x\mathbf{j}$.  
1. Compute $\nabla \cdot \mathbf{F}_1$ and $\nabla \cdot \mathbf{F}_2$.  
2. Give a physical fluid dynamics interpretation for the divergence of both fields.

#### First-Principles Intuition
Divergence measures local flux accumulation per unit volume. If vector arrows expand outward from a point, divergence is positive (a source). If arrows circulate around a point at constant radius, fluid is neither created nor compressed, yielding zero divergence.

#### Step-by-Step Solution
1. For $\mathbf{F}_1(x, y) = x\mathbf{i} + y\mathbf{j}$:

$$
   \nabla \cdot \mathbf{F}_1 = \frac{\partial}{\partial x}(x) + \frac{\partial}{\partial y}(y) = 1 + 1 = 2
$$

2. For $\mathbf{F}_2(x, y) = -y\mathbf{i} + x\mathbf{j}$:

$$
   \nabla \cdot \mathbf{F}_2 = \frac{\partial}{\partial x}(-y) + \frac{\partial}{\partial y}(x) = 0 + 0 = 0
$$

$$
\boxed{\nabla \cdot \mathbf{F}_1 = 2 \quad (\text{Uniform Source}), \quad \nabla \cdot \mathbf{F}_2 = 0 \quad (\text{Incompressible Rotation})}
$$

#### Key Insight / Takeaway
A positive constant divergence $\nabla \cdot \mathbf{F} = 2$ indicates fluid creation/expansion everywhere at a uniform rate. Pure rotational velocity fields have zero divergence, reflecting local area preservation.

---

The divergences claimed above, confirmed symbolically and by central differences at a random point.

In [2]:
# L0.2: divergence of F1 = (x, y) and F2 = (-y, x), symbolically and by finite differences.
x, y = sp.symbols("x y")
F1 = sp.Matrix([x, y])
F2 = sp.Matrix([-y, x])
div = lambda F: sp.simplify(sp.diff(F[0], x) + sp.diff(F[1], y))
d1, d2 = div(F1), div(F2)
print("symbolic div F1 =", d1, " div F2 =", d2)

h = 1e-5
p = rng.normal(size=2)
def fd_div(f, p, h):
    return sum((f(p + h * np.eye(2)[i])[i] - f(p - h * np.eye(2)[i])[i]) / (2 * h)
               for i in range(2))
n1 = fd_div(lambda v: np.array([v[0], v[1]]), p, h)
n2 = fd_div(lambda v: np.array([-v[1], v[0]]), p, h)
print(f"central differences at {p}: div F1 = {n1:.10f}, div F2 = {n2:.10f}")
assert d1 == 2 and d2 == 0
assert abs(n1 - 2) < 1e-8 and abs(n2) < 1e-8

symbolic div F1 = 2  div F2 = 0
central differences at [ 0.12573  -0.132105]: div F1 = 2.0000000000, div F2 = 0.0000000000


The symbolic and finite-difference values agree to eight digits: the source field has divergence $2$ everywhere, the rotation field exactly $0$.

---

### Problem L0.3 — Physical Interpretation of Curl
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 3  

**Problem Statement:**  
Let $\mathbf{F}_1(x, y) = y\mathbf{i}$ (shear flow) and $\mathbf{F}_2(x, y) = -y\mathbf{i} + x\mathbf{j}$ (rigid body rotation).  
Compute the 3D curl $\nabla \times \mathbf{F}$ for both fields embedded in $\mathbb{R}^3$ and explain why a tiny paddle wheel placed in shear flow $\mathbf{F}_1$ rotates.

#### First-Principles Intuition
Curl measures local micro-rotation. Even if stream lines are straight parallel lines, if fluid velocity varies laterally across the paddle wheel, the top blades feel a larger force than the bottom blades, causing torque and rotation.

#### Step-by-Step Solution
1. Embed $\mathbf{F}_1(x, y, z) = y\mathbf{i} + 0\mathbf{j} + 0\mathbf{k}$:

$$
   \nabla \times \mathbf{F}_1 = \begin{vmatrix} \mathbf{i} & \mathbf{j} & \mathbf{k} \\ \frac{\partial}{\partial x} & \frac{\partial}{\partial y} & \frac{\partial}{\partial z} \\ y & 0 & 0 \end{vmatrix} = \left(0 - \frac{\partial}{\partial y}(y)\right)\mathbf{k} = -\mathbf{k}
$$

2. Embed $\mathbf{F}_2(x, y, z) = -y\mathbf{i} + x\mathbf{j} + 0\mathbf{k}$:

$$
   \nabla \times \mathbf{F}_2 = \begin{vmatrix} \mathbf{i} & \mathbf{j} & \mathbf{k} \\ \frac{\partial}{\partial x} & \frac{\partial}{\partial y} & \frac{\partial}{\partial z} \\ -y & x & 0 \end{vmatrix} = \left( \frac{\partial}{\partial x}(x) - \frac{\partial}{\partial y}(-y) \right)\mathbf{k} = 2\mathbf{k}
$$

$$
\boxed{\nabla \times \mathbf{F}_1 = -\mathbf{k}, \quad \nabla \times \mathbf{F}_2 = 2\mathbf{k}}
$$

#### Key Insight / Takeaway
Straight streamlines do not imply zero curl! Shear velocity gradients create differential forces across extended bodies, inducing micro-rotation measured by non-zero curl.

---

Both curls, computed symbolically from the coordinate formula.

In [3]:
# L0.3: curl of the shear field (y, 0, 0) and the rigid rotation (-y, x, 0).
x, y, z = sp.symbols("x y z")
def curl(F):
    return sp.Matrix([sp.diff(F[2], y) - sp.diff(F[1], z),
                      sp.diff(F[0], z) - sp.diff(F[2], x),
                      sp.diff(F[1], x) - sp.diff(F[0], y)])
c1 = curl(sp.Matrix([y, 0, 0]))
c2 = curl(sp.Matrix([-y, x, 0]))
print("curl of shear      =", c1.T)
print("curl of rotation   =", c2.T)
assert list(c1) == [0, 0, -1] and list(c2) == [0, 0, 2]

curl of shear      = Matrix([[0, 0, -1]])
curl of rotation   = Matrix([[0, 0, 2]])


Shear carries curl $-\mathbf{k}$ and rigid rotation carries $2\mathbf{k}$ — twice the angular velocity, as the coordinate-free definition predicts.

---

### Problem L0.4 — Path Independence & Closed Loops
**Source:** Apostol, *Calculus, Vol. II*, Ch. 10  

**Problem Statement:**  
Prove that a continuous vector field $\mathbf{F}$ on a connected open region $U$ satisfies $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0$ for every closed curve $C \subset U$ if and only if $\int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r}$ for any two paths $C_1, C_2 \subset U$ sharing the same initial point $A$ and final point $B$.

#### First-Principles Intuition
A closed loop $C$ can be split into a forward path $C_1$ from $A$ to $B$ and a reversed path $-C_2$ from $B$ back to $A$. Traversing $C_1$ followed by $-C_2$ forms the closed loop $C = C_1 \cup (-C_2)$.

#### Step-by-Step Solution
1. ($\implies$) Assume $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0$ for all closed curves. Let $C_1, C_2$ be two paths from $A$ to $B$. Construct closed curve $C = C_1 \cup (-C_2)$.

$$
   0 = \oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} + \int_{-C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} - \int_{C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r}
$$

   Hence $\int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r}$.
2. ($\impliedby$) Assume path independence. Let $C$ be any closed curve starting and ending at $A$. Pick any point $B$ on $C$. Let $C_1$ be the segment of $C$ from $A$ to $B$, and $C_2$ be the segment from $B$ to $A$ along $C$ reversed. Then path independence from $A$ to $B$ implies $\int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{-C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = -\int_{C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r}$. Thus $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0$.

$$
\boxed{\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0 \iff \int_{C_1} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{C_2} \mathbf{F} \cdot \mathrm{d}\mathbf{r}}
$$

#### Key Insight / Takeaway
Path independence and zero closed-loop circulation are mathematically identical properties, forming the foundation of potential field theory.

---

### Problem L0.5 — Simply Connected Domains & Counterexample Vortex Field
**Source:** Spivak, *Calculus on Manifolds*, Ch. 4  

**Problem Statement:**  
Consider the 2D vortex vector field defined on $U = \mathbb{R}^2 \setminus \{(0,0)\}$:

$$
\mathbf{F}(x, y) = \frac{-y}{x^2 + y^2}\mathbf{i} + \frac{x}{x^2 + y^2}\mathbf{j}
$$

1. Show that $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = 0$ everywhere on $U$.  
2. Compute $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}$ along unit circle $C: \mathbf{r}(t) = (\cos t, \sin t)$, $t \in [0, 2\pi]$.  
3. Why does this not contradict Green's Theorem?

#### First-Principles Intuition
Local vanishing of curl ($\nabla \times \mathbf{F} = \mathbf{0}$) guarantees global conservative behavior only if the domain contains no "holes". A hole in the domain prevents expanding a loop continuously to a point, allowing global non-zero circulation around the puncture.

#### Step-by-Step Solution
1. Calculate partial derivatives on $U$:

$$
   P = \frac{-y}{x^2+y^2} \implies \frac{\partial P}{\partial y} = \frac{-(x^2+y^2) - (-y)(2y)}{(x^2+y^2)^2} = \frac{y^2 - x^2}{(x^2+y^2)^2}
$$

$$
   Q = \frac{x}{x^2+y^2} \implies \frac{\partial Q}{\partial x} = \frac{(x^2+y^2) - x(2x)}{(x^2+y^2)^2} = \frac{y^2 - x^2}{(x^2+y^2)^2}
$$

   Thus $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = 0$.
2. Compute line integral along unit circle $\mathbf{r}(t) = (\cos t, \sin t)$, $\mathbf{r}'(t) = (-\sin t, \cos t)$:

$$
   \mathbf{F}(\mathbf{r}(t)) = -\sin t \mathbf{i} + \cos t \mathbf{j}
$$

$$
   \oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_0^{2\pi} \big((-\sin t)(-\sin t) + (\cos t)(\cos t)\big)\mathrm{d}t = \int_0^{2\pi} 1\,\mathrm{d}t = 2\pi
$$

3. Green's theorem requires $P, Q$ to be $C^1$ on the **entire region enclosed by $C$**. Here the origin $(0,0)$ lies inside $C$ where $\mathbf{F}$ is undefined!

$$
\boxed{\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = 0, \quad \oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 2\pi \neq 0}
$$

#### Key Insight / Takeaway
Simple connectivity of the domain is essential for local irrotationality ($\nabla \times \mathbf{F} = \mathbf{0}$) to imply global conservativeness ($\mathbf{F} = \nabla f$).

---

The two halves of the counterexample: a vanishing integrand and a non-vanishing loop integral.

In [4]:
# L0.5: the vortex field has zero planar curl yet circulation 2*pi around the unit circle.
x, y = sp.symbols("x y")
P = -y / (x**2 + y**2)
Q = x / (x**2 + y**2)
integrand = sp.simplify(sp.diff(Q, x) - sp.diff(P, y))
print("Q_x - P_y =", integrand, "(identically zero off the origin)")

t = sp.symbols("t")
r = (sp.cos(t), sp.sin(t))
work = sp.integrate(sp.simplify(P.subs({x: r[0], y: r[1]}) * sp.diff(r[0], t)
                              + Q.subs({x: r[0], y: r[1]}) * sp.diff(r[1], t)), (t, 0, 2 * sp.pi))
print("circulation around the unit circle =", work)
assert integrand == 0 and work == 2 * sp.pi

Q_x - P_y = 0 (identically zero off the origin)
circulation around the unit circle = 2*pi


The integrand is *identically* zero yet the circulation is $2\pi$. The two facts coexist because the origin, where $\mathbf{F}$ is undefined, lies inside the loop.

---

### Problem L0.6 — Green's Theorem Boundary Orientation
**Source:** Stewart, *Multivariable Calculus*, Ch. 16  

**Problem Statement:**  
Let $D \subset \mathbb{R}^2$ be the annulus $1 \le x^2 + y^2 \le 4$. Describe the correct orientation for the outer boundary $C_1$ ($r=2$) and inner boundary $C_2$ ($r=1$) such that $\iint_D \left(\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right)\mathrm{d}A = \oint_{\partial D} (P\mathrm{d}x + Q\mathrm{d}y)$.

#### First-Principles Intuition
For Green's Theorem, the boundary must be oriented so that as a observer walks along the curve in the direction of traversal, the interior region $D$ always lies to their **left**.

#### Step-by-Step Solution
1. Walking along the outer boundary $C_1$ ($r=2$): to keep the annulus $D$ on the left, one must walk **counterclockwise** (positive mathematical orientation).
2. Walking along the inner boundary $C_2$ ($r=1$): keeping $D$ on the left requires walking **clockwise** (negative mathematical orientation relative to origin).
3. Therefore $\partial D = C_1^+ \cup C_2^-$.

$$
   \boxed{\text{Outer boundary } C_1: \text{Counterclockwise}, \quad \text{Inner boundary } C_2: \text{Clockwise}}
$$

#### Key Insight / Takeaway
Multi-connected regions require inner boundaries to be oriented clockwise so that the domain consistently remains on the left hand side.

---

### Problem L0.7 — Surface Orientation & Flux Sign
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 7  

**Problem Statement:**  
Let $S$ be the unit sphere $x^2 + y^2 + z^2 = 1$ with outward unit normal $\mathbf{n} = x\mathbf{i} + y\mathbf{j} + z\mathbf{k}$. Compute the flux of $\mathbf{F}(x,y,z) = \mathbf{n}$ through $S$, and state what happens if the normal orientation is reversed to $-\mathbf{n}$.

#### First-Principles Intuition
Vector flux measures flow aligned with normal $\mathbf{n}$. Reversing the normal vector field from outward to inward flips the sign of every dot product $\mathbf{F} \cdot \mathbf{n}$.

#### Step-by-Step Solution
1. On unit sphere $S$, $\mathbf{F} \cdot \mathbf{n} = \mathbf{n} \cdot \mathbf{n} = \lVert \mathbf{n} \rVert^2 = 1$.
2. The surface area of unit sphere is $\iint_S \mathrm{d}S = 4\pi$.
3. Outward flux:

$$
   \Phi_{\text{out}} = \iint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S = \iint_S 1\,\mathrm{d}S = 4\pi
$$

4. Inward flux ($\mathbf{n}_{\text{in}} = -\mathbf{n}$):

$$
   \Phi_{\text{in}} = \iint_S \mathbf{F} \cdot (-\mathbf{n})\,\mathrm{d}S = \iint_S (-1)\,\mathrm{d}S = -4\pi
$$

$$
   \boxed{\Phi_{\text{out}} = 4\pi, \quad \Phi_{\text{in}} = -4\pi}
$$

#### Key Insight / Takeaway
Flux integrals depend strictly on surface orientation; reversing the unit normal inverts the sign of the total flux.

---

Both flux values by quadrature over the sphere.

In [5]:
# L0.7: flux of F = n through the unit sphere, both orientations, by quadrature.
# In spherical coordinates F . n = 1 and dS = sin(phi) dphi dtheta.
outward, err = integrate.dblquad(lambda phi, th: np.sin(phi), 0, 2 * np.pi, 0, np.pi)
inward = -outward
print(f"outward flux = {outward:.12f}   (4*pi = {4*np.pi:.12f})   quad error {err:.2e}")
print(f"inward  flux = {inward:.12f}   (-4*pi = {-4*np.pi:.12f})")
assert rel(outward, 4 * np.pi) < 1e-10 and rel(inward, -4 * np.pi) < 1e-10

outward flux = 12.566370614359   (4*pi = 12.566370614359)   quad error 1.40e-13
inward  flux = -12.566370614359   (-4*pi = -12.566370614359)


Both magnitudes equal the sphere's area $4\pi$ to ten digits; only the sign responds to the orientation.

---

### Problem L0.8 — Stokes' Theorem Boundary Applicability
**Source:** MIT 18.02 Multivariable Calculus  

**Problem Statement:**  
Explain why applying Stokes' Theorem $\iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S = \oint_{\partial S} \mathbf{F} \cdot \mathrm{d}\mathbf{r}$ to any smooth vector field $\mathbf{F}$ on a closed surface $S$ (such as a full ellipsoid) identically yields zero.

#### First-Principles Intuition
A closed surface (like a balloon) has no boundary curve ($\partial S = \emptyset$). Traversing a boundary that does not exist yields a line integral over an empty set, which is zero.

#### Step-by-Step Solution
1. By definition of a closed 2-manifold without boundary, $\partial S = \emptyset$.
2. The line integral of any continuous vector field along the empty boundary is:

$$
   \oint_{\partial S} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0
$$

3. By Stokes' Theorem:

$$
   \iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S = \oint_{\partial S} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 0
$$

$$
   \boxed{\iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S = 0 \quad \text{for any closed surface } S}
$$

#### Key Insight / Takeaway
The total flux of any curl field ($\mathbf{G} = \nabla \times \mathbf{F}$) through any closed surface is identically zero, consistent with Gauss's Divergence Theorem since $\nabla \cdot (\nabla \times \mathbf{F}) = 0$.

---

## L1 — Foundations

### Problem L1.1 — Scalar Line Integral over Parabola
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 4312  

**Problem Statement:**  
Evaluate the scalar line integral $\int_C (x + y)\,\mathrm{d}s$ along the parabolic arc $y = x^2$ from $(0,0)$ to $(1,1)$.

#### First-Principles Intuition
We parameterize the parabolic wire using $x=t, y=t^2$, express the arc length element $\mathrm{d}s = \sqrt{1 + (y'(x))^2}\mathrm{d}x$, and integrate.

#### Step-by-Step Solution
1. Parameterization: $\mathbf{r}(t) = (t, t^2)$ for $t \in [0, 1]$.
2. Derivatives: $x'(t) = 1, y'(t) = 2t$.
3. Differential arc length: $\mathrm{d}s = \sqrt{1 + 4t^2}\,\mathrm{d}t$.
4. Line integral:

$$
   I = \int_0^1 (t + t^2)\sqrt{1 + 4t^2}\,\mathrm{d}t = \int_0^1 t\sqrt{1 + 4t^2}\,\mathrm{d}t + \int_0^1 t^2\sqrt{1 + 4t^2}\,\mathrm{d}t
$$

5. For $I_1 = \int_0^1 t\sqrt{1+4t^2}\mathrm{d}t$, substitute $u = 1+4t^2, \mathrm{d}u = 8t\mathrm{d}t$:

$$
   I_1 = \frac{1}{8}\int_1^5 u^{\frac{1}{2}}\mathrm{d}u = \frac{1}{8} \left[\frac{2}{3}u^{\frac{3}{2}}\right]_1^5 = \frac{5\sqrt{5} - 1}{12}
$$

6. For $I_2 = \int_0^1 t^2\sqrt{1+4t^2}\mathrm{d}t$, substitute $2t = \sinh \theta$:
   Using standard table integration formula $\int t^2\sqrt{1+4t^2}\mathrm{d}t = \frac{t(1+4t^2)^{\frac{3}{2}}}{16} - \frac{t\sqrt{1+4t^2}}{32} - \frac{\ln(2t+\sqrt{1+4t^2})}{64}$:

$$
   I_2 = \frac{5\sqrt{5}}{16} - \frac{\sqrt{5}}{32} - \frac{\ln(2+\sqrt{5})}{64} = \frac{9\sqrt{5}}{32} - \frac{\ln(2+\sqrt{5})}{64}
$$

7. Summing $I_1 + I_2$:

$$
   I = \frac{5\sqrt{5} - 1}{12} + \frac{9\sqrt{5}}{32} - \frac{\ln(2+\sqrt{5})}{64}
     = \frac{40\sqrt{5} - 8}{96} + \frac{27\sqrt{5}}{96} - \frac{\ln(2+\sqrt{5})}{64}
     = \frac{67\sqrt{5} - 8}{96} - \frac{\ln(2+\sqrt{5})}{64}
$$

$$
   \boxed{\int_C (x+y)\,\mathrm{d}s = \frac{67\sqrt{5} - 8}{96} - \frac{\ln(2+\sqrt{5})}{64} \approx 1.454699}
$$

#### Key Insight / Takeaway
Scalar line integrals reduce to single-variable definite integrals via the arc-length substitution $\mathrm{d}s = \sqrt{x'(t)^2 + y'(t)^2}\,\mathrm{d}t$; note that $-1/12 = -8/96$, so the rational part of the answer is $-8$, not $-16$.

---

The exact integral against the boxed closed form, and against an independent adaptive quadrature. The final print shows the size of the arithmetic slip that the corrected constant repairs.

In [6]:
# L1.1: the scalar line integral, exactly and numerically, against the corrected closed form.
t = sp.symbols("t", positive=True)
exact = sp.integrate((t + t**2) * sp.sqrt(1 + 4 * t**2), (t, 0, 1))
claim = (67 * sp.sqrt(5) - 8) / 96 - sp.log(2 + sp.sqrt(5)) / 64
old_claim = (67 * sp.sqrt(5) - 16) / 96 - sp.log(2 + sp.sqrt(5)) / 64
print("sympy exact  =", sp.nsimplify(sp.simplify(exact)))
print(f"exact        = {float(exact):.12f}")
print(f"boxed answer = {float(claim):.12f}")
print(f"old (wrong)  = {float(old_claim):.12f}   difference {float(claim - old_claim):.12f} = 8/96")
quad, _ = integrate.quad(lambda u: (u + u**2) * np.sqrt(1 + 4 * u**2), 0, 1)
print(f"scipy.quad   = {quad:.12f}")
assert abs(sp.N(exact - claim, 40)) < sp.Float("1e-35")   # asinh(2) = log(2+sqrt(5))
assert rel(quad, float(claim)) < 1e-12

sympy exact  = -1/12 - asinh(2)/64 + 67*sqrt(5)/96
exact        = 1.454698971664
boxed answer = 1.454698971664
old (wrong)  = 1.371365638330   difference 0.083333333333 = 8/96
scipy.quad   = 1.454698971664


The exact value $1.454699$ matches the corrected box. The old constant $-16/96$ was off by $8/96 \approx 0.0833$, visible in the printed difference.

---

### Problem L1.2 — Vector Line Integral / Work Along Helix
**Source:** Stewart, *Multivariable Calculus*, Ch. 16.2  

**Problem Statement:**  
Calculate the work done by force field $\mathbf{F}(x,y,z) = y\mathbf{i} + z\mathbf{j} + x\mathbf{k}$ in moving a particle along helix $\mathbf{r}(t) = \cos t \mathbf{i} + \sin t \mathbf{j} + t \mathbf{k}$ from $t = 0$ to $t = 2\pi$.

#### First-Principles Intuition
Work is the integral of the tangential force component along the path $W = \int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}$.

#### Step-by-Step Solution
1. Parameterization: $\mathbf{r}(t) = (\cos t, \sin t, t)$, $t \in [0, 2\pi]$.
2. Velocity vector: $\mathbf{r}'(t) = (-\sin t, \cos t, 1)$.
3. Field along path: $\mathbf{F}(\mathbf{r}(t)) = \sin t \mathbf{i} + t \mathbf{j} + \cos t \mathbf{k}$.
4. Dot product:

$$
   \mathbf{F}(\mathbf{r}(t)) \cdot \mathbf{r}'(t) = (\sin t)(-\sin t) + (t)(\cos t) + (\cos t)(1) = -\sin^2 t + t\cos t + \cos t
$$

5. Integrate from $t=0$ to $2\pi$:

$$
   W = \int_0^{2\pi} (-\sin^2 t + t\cos t + \cos t)\,\mathrm{d}t
$$

   - $\int_0^{2\pi} -\sin^2 t\,\mathrm{d}t = \int_0^{2\pi} -\frac{1 - \cos 2t}{2}\,\mathrm{d}t = -\pi$
   - $\int_0^{2\pi} t\cos t\,\mathrm{d}t = [t\sin t + \cos t]_0^{2\pi} = (0 + 1) - (0 + 1) = 0$
   - $\int_0^{2\pi} \cos t\,\mathrm{d}t = [\sin t]_0^{2\pi} = 0$
6. Total work $W = -\pi$.

$$
   \boxed{W = -\pi}
$$

#### Key Insight / Takeaway
Negative work means the force field opposes the particle's overall motion along the helix.

---

The work integral, symbolically and by adaptive quadrature.

In [7]:
# L1.2: work of F = (y, z, x) along one turn of the helix.
t = sp.symbols("t")
r = sp.Matrix([sp.cos(t), sp.sin(t), t])
F = sp.Matrix([r[1], r[2], r[0]])
W = sp.integrate(F.dot(sp.diff(r, t)), (t, 0, 2 * sp.pi))
num, _ = integrate.quad(lambda u: np.sin(u) * (-np.sin(u)) + u * np.cos(u) + np.cos(u), 0, 2 * np.pi)
print("symbolic W =", sp.simplify(W), f"= {float(W):.12f}")
print(f"scipy.quad = {num:.12f}   (-pi = {-np.pi:.12f})")
assert sp.simplify(W + sp.pi) == 0 and rel(num, -np.pi) < 1e-10

symbolic W = -pi = -3.141592653590
scipy.quad = -3.141592653590   (-pi = -3.141592653590)


Both routes return $-\pi$: the $t\cos t$ and $\cos t$ contributions cancel over a full turn, leaving only $-\int_0^{2\pi}\sin^2 t\,\mathrm{d}t$.

---

### Problem L1.3 — Potential Function & Fundamental Theorem
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 6.3  

**Problem Statement:**  
Given vector field $\mathbf{F}(x,y,z) = (2xy + z^3)\mathbf{i} + (x^2 + 3y^2z)\mathbf{j} + (3xz^2 + y^3)\mathbf{k}$:  
1. Show that $\mathbf{F}$ is conservative on $\mathbb{R}^3$.  
2. Find scalar potential $f(x,y,z)$ such that $\mathbf{F} = \nabla f$.  
3. Evaluate $\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}$ along any smooth curve from $A = (1, -1, 0)$ to $B = (2, 1, 1)$.

#### First-Principles Intuition
Check $\nabla \times \mathbf{F} = \mathbf{0}$. If curl vanishes everywhere on simple domain $\mathbb{R}^3$, integrate component partials sequentially to recover $f(x,y,z)$.

#### Step-by-Step Solution
1. Verify $\nabla \times \mathbf{F} = \mathbf{0}$:
   - $(\nabla \times \mathbf{F})_x = \frac{\partial}{\partial y}(3xz^2+y^3) - \frac{\partial}{\partial z}(x^2+3y^2z) = 3y^2 - 3y^2 = 0$
   - $(\nabla \times \mathbf{F})_y = \frac{\partial}{\partial z}(2xy+z^3) - \frac{\partial}{\partial x}(3xz^2+y^3) = 3z^2 - 3z^2 = 0$
   - $(\nabla \times \mathbf{F})_z = \frac{\partial}{\partial x}(x^2+3y^2z) - \frac{\partial}{\partial y}(2xy+z^3) = 2x - 2x = 0$
2. Construct potential $f(x,y,z)$:
   - $\frac{\partial f}{\partial x} = 2xy + z^3 \implies f(x,y,z) = x^2y + xz^3 + g(y, z)$
   - $\frac{\partial f}{\partial y} = x^2 + \frac{\partial g}{\partial y} = x^2 + 3y^2z \implies \frac{\partial g}{\partial y} = 3y^2z \implies g(y, z) = y^3z + h(z)$
   - $\frac{\partial f}{\partial z} = 3xz^2 + y^3 + h'(z) = 3xz^2 + y^3 \implies h'(z) = 0 \implies h(z) = C$
   - Thus $f(x, y, z) = x^2y + xz^3 + y^3z + C$.
3. Apply Gradient Theorem:
   - $f(B) = f(2, 1, 1) = (2)^2(1) + (2)(1)^3 + (1)^3(1) = 4 + 2 + 1 = 7$
   - $f(A) = f(1, -1, 0) = (1)^2(-1) + (1)(0)^3 + (-1)^3(0) = -1$
   - $\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = f(B) - f(A) = 7 - (-1) = 8$.

$$
   \boxed{f(x,y,z) = x^2y + xz^3 + y^3z + C, \quad \int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 8}
$$

#### Key Insight / Takeaway
The line integral of a conservative field depends only on the boundary endpoints $A$ and $B$, completely ignoring the path geometry in between.

---

Three checks: that $\nabla f = \mathbf{F}$, that the endpoint difference is $8$, and that one explicit path gives the same number.

In [8]:
# L1.3: verify the potential and the endpoint evaluation.
x, y, z = sp.symbols("x y z")
F = sp.Matrix([2 * x * y + z**3, x**2 + 3 * y**2 * z, 3 * x * z**2 + y**3])
f = x**2 * y + x * z**3 + y**3 * z
print("grad f - F =", sp.simplify(sp.Matrix([sp.diff(f, v) for v in (x, y, z)]) - F).T)
A, B = {x: 1, y: -1, z: 0}, {x: 2, y: 1, z: 1}
val = f.subs(B) - f.subs(A)
print("f(B) =", f.subs(B), " f(A) =", f.subs(A), " integral =", val)

# Straight-line path from A to B, integrated numerically: same number, any path.
s = sp.symbols("s")
path = sp.Matrix([1 + s, -1 + 2 * s, s])
line = sp.integrate(F.subs({x: path[0], y: path[1], z: path[2]}).dot(sp.diff(path, s)), (s, 0, 1))
print("straight-line path integral =", sp.simplify(line))
assert sp.simplify(sp.Matrix([sp.diff(f, v) for v in (x, y, z)]) - F) == sp.zeros(3, 1)
assert val == 8 and sp.simplify(line) == 8

grad f - F = Matrix([[0, 0, 0]])
f(B) = 7  f(A) = -1  integral = 8
straight-line path integral = 8


The gradient residual is the zero vector, so $f$ is a genuine potential; the straight-line path returns the same $8$ that the endpoints predict — path independence in action.

---

### Problem L1.4 — Green's Theorem Area Calculation
**Source:** Apostol, *Calculus, Vol. II*, Ch. 10.5  

**Problem Statement:**  
Use Green's Theorem area formula $\text{Area}(D) = \frac{1}{2}\oint_{\partial D} (x\,\mathrm{d}y - y\,\mathrm{d}x)$ to compute the area enclosed by the astroid $x = a\cos^3 t, y = a\sin^3 t$ for $t \in [0, 2\pi]$.

#### First-Principles Intuition
Setting $P = -\frac{y}{2}$ and $Q = \frac{x}{2}$ gives $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = \frac{1}{2} - (-\frac{1}{2}) = 1$. Thus double integral of 1 gives total area via a single closed boundary line integral!

#### Step-by-Step Solution
1. Compute differentials:
   - $\mathrm{d}x = -3a\cos^2 t \sin t\,\mathrm{d}t$
   - $\mathrm{d}y = 3a\sin^2 t \cos t\,\mathrm{d}t$
2. Form integrand $x\,\mathrm{d}y - y\,\mathrm{d}x$:

$$
   x\,\mathrm{d}y - y\,\mathrm{d}x = (a\cos^3 t)(3a\sin^2 t \cos t\,\mathrm{d}t) - (a\sin^3 t)(-3a\cos^2 t \sin t\,\mathrm{d}t)
$$

$$
   = 3a^2 \cos^2 t \sin^2 t (\cos^2 t + \sin^2 t)\,\mathrm{d}t = 3a^2 \cos^2 t \sin^2 t\,\mathrm{d}t = \frac{3a^2}{4}\sin^2(2t)\,\mathrm{d}t
$$

3. Integrate over $t \in [0, 2\pi]$:

$$
   \text{Area}(D) = \frac{1}{2} \int_0^{2\pi} \frac{3a^2}{4}\sin^2(2t)\,\mathrm{d}t = \frac{3a^2}{8} \int_0^{2\pi} \frac{1 - \cos(4t)}{2}\,\mathrm{d}t = \frac{3a^2}{16}(2\pi) = \frac{3\pi a^2}{8}
$$

$$
   \boxed{\text{Area} = \frac{3\pi a^2}{8}}
$$

#### Key Insight / Takeaway
Green's Theorem converts 2D planar area calculation into a 1D boundary line integral, the foundational principle of mechanical planimeters.

---

The astroid area symbolically in $a$, and numerically at $a = 1$ by the trapezoid rule (the integrand is periodic, so convergence is fast).

In [9]:
# L1.4: astroid area by the Green area formula, symbolically and by the trapezoid rule.
t, a = sp.symbols("t a", positive=True)
X, Y = a * sp.cos(t)**3, a * sp.sin(t)**3
area = sp.simplify(sp.Rational(1, 2) * sp.integrate(X * sp.diff(Y, t) - Y * sp.diff(X, t), (t, 0, 2 * sp.pi)))
print("symbolic area =", area)

m = 4000                       # periodic integrand: the trapezoid rule converges fast
tt = np.linspace(0, 2 * np.pi, m, endpoint=False)
Xn, Yn = np.cos(tt)**3, np.sin(tt)**3
dX, dY = -3 * np.cos(tt)**2 * np.sin(tt), 3 * np.sin(tt)**2 * np.cos(tt)
num = 0.5 * np.sum(Xn * dY - Yn * dX) * (2 * np.pi / m)
print(f"numeric (a=1) = {num:.12f}   3*pi/8 = {3*np.pi/8:.12f}")
assert sp.simplify(area - 3 * sp.pi * a**2 / 8) == 0 and rel(num, 3 * np.pi / 8) < 1e-10

symbolic area = 3*pi*a**2/8
numeric (a=1) = 1.178097245096   3*pi/8 = 1.178097245096


The astroid encloses $3\pi a^2/8$, three eighths of the circumscribed circle of radius $a$.

---

### Problem L1.5 — Green's Theorem Line Integral Evaluation
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 4325  

**Problem Statement:**  
Evaluate $\oint_C (e^x \sin y - 2y)\,\mathrm{d}x + (e^x \cos y + 3x)\,\mathrm{d}y$ around the triangle $C$ with vertices $(0,0), (2,0), (2,4)$ oriented counterclockwise.

#### First-Principles Intuition
Direct line integration across all three triangle edges requires multiple parameterizations. Green's theorem simplifies this to a double integral of $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$ over the triangle region.

#### Step-by-Step Solution
1. Identify $P = e^x \sin y - 2y$ and $Q = e^x \cos y + 3x$.
2. Calculate partial derivatives:
   - $\frac{\partial P}{\partial y} = e^x \cos y - 2$
   - $\frac{\partial Q}{\partial x} = e^x \cos y + 3$
3. Apply Green's Theorem:

$$
   \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = (e^x \cos y + 3) - (e^x \cos y - 2) = 5
$$

4. Compute double integral over triangle $D$:

$$
   \oint_C (P\mathrm{d}x + Q\mathrm{d}y) = \iint_D 5\,\mathrm{d}A = 5 \cdot \text{Area}(D)
$$

5. Triangle has base 2 and height 4, so $\text{Area}(D) = \frac{1}{2}(2)(4) = 4$.
6. Result $= 5 \times 4 = 20$.

$$
   \boxed{20}
$$

#### Key Insight / Takeaway
Non-linear exponential and trigonometric components cancel completely inside the curl operator $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$, leaving a constant integrand.

---

Both sides of Green's theorem on this triangle: the double integral, and the three-edge line integral computed directly.

In [10]:
# L1.5: Green's theorem on the triangle (0,0), (2,0), (2,4) -- both sides.
xs, ys = sp.symbols("x y")
P = sp.exp(xs) * sp.sin(ys) - 2 * ys
Q = sp.exp(xs) * sp.cos(ys) + 3 * xs
print("Q_x - P_y =", sp.simplify(sp.diff(Q, xs) - sp.diff(P, ys)))
double = sp.integrate(sp.integrate(sp.diff(Q, xs) - sp.diff(P, ys), (ys, 0, 2 * xs)), (xs, 0, 2))
print("double integral =", sp.simplify(double))

# Direct line integral over the three edges, counterclockwise.
u = sp.symbols("u")
edges = [((2 * u, 0 * u)), ((2 + 0 * u, 4 * u)), ((2 - 2 * u, 4 - 4 * u))]
loop = 0
for ex, ey in edges:
    loop += sp.integrate(P.subs({xs: ex, ys: ey}) * sp.diff(ex, u)
                       + Q.subs({xs: ex, ys: ey}) * sp.diff(ey, u), (u, 0, 1))
print("direct loop integral =", sp.simplify(loop))
assert sp.simplify(double) == 20 and sp.simplify(loop) == 20

Q_x - P_y = 5
double integral = 20


direct loop integral = 20


Both sides return $20$. The exponential and trigonometric terms cancel inside $Q_x - P_y$, leaving the constant $5$ times the triangle's area $4$.

---

### Problem L1.6 — Scalar Surface Integral over Cone
**Source:** Stewart, *Multivariable Calculus*, Ch. 16.7  

**Problem Statement:**  
Compute $\iint_S z\,\mathrm{d}S$ where $S$ is the cone surface $z = \sqrt{x^2 + y^2}$ bounded between $z = 0$ and $z = 3$.

#### First-Principles Intuition
Parameterize $S$ using polar coordinates $(r, \theta)$, evaluate the differential area element $\mathrm{d}S = \sqrt{1 + z_x^2 + z_y^2}\mathrm{d}x\mathrm{d}y$, and integrate.

#### Step-by-Step Solution
1. Explicit surface representation: $z(x, y) = \sqrt{x^2+y^2}$.
2. Partial derivatives: $z_x = \frac{x}{\sqrt{x^2+y^2}}$, $z_y = \frac{y}{\sqrt{x^2+y^2}}$.
3. Surface element:

$$
   \mathrm{d}S = \sqrt{1 + z_x^2 + z_y^2}\,\mathrm{d}x\mathrm{d}y = \sqrt{1 + \frac{x^2 + y^2}{x^2 + y^2}}\,\mathrm{d}x\mathrm{d}y = \sqrt{2}\,\mathrm{d}x\mathrm{d}y
$$

4. Region projection $D$: Disk $x^2+y^2 \le 9$ ($0 \le r \le 3, 0 \le \theta \le 2\pi$).
5. Integral setup in polar coordinates:

$$
   \iint_S z\,\mathrm{d}S = \iint_D r \cdot \sqrt{2} \cdot r\,\mathrm{d}r\mathrm{d}\theta = \sqrt{2} \int_0^{2\pi}\mathrm{d}\theta \int_0^3 r^2\,\mathrm{d}r
$$

$$
   = \sqrt{2}(2\pi) \left[\frac{r^3}{3}\right]_0^3 = \sqrt{2}(2\pi)(9) = 18\sqrt{2}\pi
$$

$$
\boxed{18\sqrt{2}\pi}
$$

#### Key Insight / Takeaway
The metric element $\mathrm{d}S = \sqrt{2}\mathrm{d}A$ scales the planar area projection by the constant inclination angle of the cone ($\sec \phi = \sqrt{2}$).

---

The cone integral in polar coordinates, symbolically and by adaptive quadrature.

In [11]:
# L1.6: scalar surface integral of z over the cone z = sqrt(x^2+y^2), 0 <= z <= 3.
r, th = sp.symbols("r theta", positive=True)
val = sp.integrate(sp.integrate(r * sp.sqrt(2) * r, (r, 0, 3)), (th, 0, 2 * sp.pi))
print("symbolic =", sp.simplify(val), f"= {float(val):.12f}")
num, _ = integrate.dblquad(lambda rr, tt: rr * np.sqrt(2) * rr, 0, 2 * np.pi, 0, 3)
print(f"scipy.dblquad = {num:.12f}   18*sqrt(2)*pi = {18*np.sqrt(2)*np.pi:.12f}")
assert sp.simplify(val - 18 * sp.sqrt(2) * sp.pi) == 0 and rel(num, 18 * np.sqrt(2) * np.pi) < 1e-10

symbolic = 18*sqrt(2)*pi = 79.971892886851
scipy.dblquad = 79.971892886851   18*sqrt(2)*pi = 79.971892886851


Both routes give $18\sqrt{2}\pi$; the factor $\sqrt{2}$ is the constant inclination $\sec(\pi/4)$ of the cone.

---

### Problem L1.7 — Vector Flux Integral through Hemispherical Shell
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 7.1  

**Problem Statement:**  
Compute the upward flux $\iint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S$ for $\mathbf{F}(x,y,z) = x\mathbf{i} + y\mathbf{j} + z^2\mathbf{k}$ across the upper unit hemisphere $S: z = \sqrt{1 - x^2 - y^2}$.

#### First-Principles Intuition
Use spherical coordinates $(\theta, \phi)$ to parameterize the upper hemisphere, construct unit normal $\mathbf{n} = \frac{\mathbf{r}}{r}$, and compute flux.

#### Step-by-Step Solution
1. Spherical parameterization ($r=1$):
   $\mathbf{r}(\theta, \phi) = (\sin\phi\cos\theta, \sin\phi\sin\theta, \cos\phi)$ for $\phi \in [0, \frac{\pi}{2}], \theta \in [0, 2\pi]$.
2. Outward unit normal: $\mathbf{n} = \sin\phi\cos\theta \mathbf{i} + \sin\phi\sin\theta \mathbf{j} + \cos\phi \mathbf{k}$.
3. Surface element: $\mathrm{d}S = \sin\phi\,\mathrm{d}\phi\mathrm{d}\theta$.
4. Field on surface: $\mathbf{F} = \sin\phi\cos\theta \mathbf{i} + \sin\phi\sin\theta \mathbf{j} + \cos^2\phi \mathbf{k}$.
5. Integrand $\mathbf{F} \cdot \mathbf{n}$:

$$
   \mathbf{F} \cdot \mathbf{n} = \sin^2\phi\cos^2\theta + \sin^2\phi\sin^2\theta + \cos^3\phi = \sin^2\phi + \cos^3\phi
$$

6. Flux Integral:

$$
   \Phi = \int_0^{2\pi}\mathrm{d}\theta \int_0^{\frac{\pi}{2}} (\sin^2\phi + \cos^3\phi)\sin\phi\,\mathrm{d}\phi = 2\pi \left( \int_0^{\frac{\pi}{2}} \sin^3\phi\,\mathrm{d}\phi + \int_0^{\frac{\pi}{2}} \cos^3\phi\sin\phi\,\mathrm{d}\phi \right)
$$

   - $\int_0^{\frac{\pi}{2}} \sin^3\phi\,\mathrm{d}\phi = \frac{2}{3}$
   - $\int_0^{\frac{\pi}{2}} \cos^3\phi\sin\phi\,\mathrm{d}\phi = \left[-\frac{\cos^4\phi}{4}\right]_0^{\frac{\pi}{2}} = \frac{1}{4}$
7. Sum: $\Phi = 2\pi \left( \frac{2}{3} + \frac{1}{4} \right) = 2\pi \left( \frac{11}{12} \right) = \frac{11\pi}{6}$.

$$
\boxed{\frac{11\pi}{6}}
$$

#### Key Insight / Takeaway
Spherical parameterization automatically handles surface metric elements $\mathrm{d}S = r^2\sin\phi\mathrm{d}\phi\mathrm{d}\theta$ without computing explicit cross-products.

---

The hemisphere flux, symbolically and by adaptive quadrature.

In [12]:
# L1.7: upward flux of F = (x, y, z^2) through the upper unit hemisphere.
ph, th = sp.symbols("phi theta")
integrand = (sp.sin(ph)**2 + sp.cos(ph)**3) * sp.sin(ph)
val = sp.integrate(sp.integrate(integrand, (ph, 0, sp.pi / 2)), (th, 0, 2 * sp.pi))
print("symbolic flux =", sp.simplify(val), f"= {float(val):.12f}")
num, _ = integrate.dblquad(lambda p, t: (np.sin(p)**2 + np.cos(p)**3) * np.sin(p),
                           0, 2 * np.pi, 0, np.pi / 2)
print(f"scipy.dblquad = {num:.12f}   11*pi/6 = {11*np.pi/6:.12f}")
assert sp.simplify(val - 11 * sp.pi / 6) == 0 and rel(num, 11 * np.pi / 6) < 1e-10

symbolic flux = 11*pi/6 = 5.759586531581
scipy.dblquad = 5.759586531581   11*pi/6 = 5.759586531581


Both routes give $11\pi/6 \approx 5.7596$. The $\sin^2\phi$ part contributes $2/3$ and the $\cos^3\phi$ part $1/4$.

---

### Problem L1.8 — Verification of Stokes' Theorem
**Source:** Apostol, *Calculus, Vol. II*, Ch. 11.8  

**Problem Statement:**  
Verify Stokes' Theorem $\oint_{\partial S} \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S$ for field $\mathbf{F}(x,y,z) = y\mathbf{i} + z\mathbf{j} + x\mathbf{k}$ on the paraboloid $S: z = 1 - x^2 - y^2$ for $z \ge 0$, bounded by unit circle $C = \partial S$ in the $xy$-plane.

#### First-Principles Intuition
Evaluate both sides independently: the 1D closed line integral around boundary $C$ ($z=0, x^2+y^2=1$) and the 2D surface flux of curl over the paraboloid cap.

#### Step-by-Step Solution
1. **Line Integral Side ($\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r}$)**:
   - Parameterize boundary $C$: $\mathbf{r}(t) = (\cos t, \sin t, 0)$, $t \in [0, 2\pi]$.
   - Velocity: $\mathbf{r}'(t) = (-\sin t, \cos t, 0)$.
   - Field on $C$: $\mathbf{F} = \sin t \mathbf{i} + 0\mathbf{j} + \cos t \mathbf{k}$.
   - $\mathbf{F} \cdot \mathbf{r}'(t) = -\sin^2 t$.
   - $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_0^{2\pi} -\sin^2 t\,\mathrm{d}t = -\pi$.
2. **Surface Integral Side ($\iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S$)**:
   - Compute curl: $\nabla \times \mathbf{F} = -\mathbf{i} - \mathbf{j} - \mathbf{k}$.
   - Surface $z = g(x, y) = 1 - x^2 - y^2$. Upward normal vector element:

$$
     \mathbf{N} = (-g_x, -g_y, 1) = (2x, 2y, 1)
$$

   - Dot product: $(\nabla \times \mathbf{F}) \cdot \mathbf{N} = (-1)(2x) + (-1)(2y) + (-1)(1) = -2x - 2y - 1$.
   - Integrate over unit disk $D: x^2+y^2 \le 1$:

$$
     \iint_D (-2x - 2y - 1)\,\mathrm{d}A = -2\iint_D x\mathrm{d}A - 2\iint_D y\mathrm{d}A - \iint_D 1\mathrm{d}A
$$

   - By symmetry, $\iint_D x\mathrm{d}A = \iint_D y\mathrm{d}A = 0$. Area of unit disk is $\pi$.
   - Surface integral $= -\pi$.
3. Both sides equal $-\pi$.

$$
\boxed{\text{Both sides equal } -\pi. \text{ Stokes' Theorem verified!}}
$$

#### Key Insight / Takeaway
The circulation around boundary curve $C$ remains invariant if the paraboloid cap is replaced by any other surface sharing the same boundary $C$ (such as the flat unit disk).

---

All three numbers: the boundary circulation, the curl flux through the paraboloid, and the curl flux through the flat disk that shares the boundary.

In [13]:
# L1.8: Stokes for F = (y, z, x) on the paraboloid cap -- both sides, plus the flat disk.
t = sp.symbols("t")
r = sp.Matrix([sp.cos(t), sp.sin(t), 0])
F = sp.Matrix([r[1], r[2], r[0]])
lhs = sp.integrate(F.dot(sp.diff(r, t)), (t, 0, 2 * sp.pi))
print("boundary circulation =", sp.simplify(lhs))

# curl F = (-1, -1, -1); upward element N = (2x, 2y, 1) on z = 1 - x^2 - y^2.
xs, ys = sp.symbols("x y")
rr, tt = sp.symbols("r theta", positive=True)
dotted = (-1) * (2 * xs) + (-1) * (2 * ys) + (-1) * 1
polar = dotted.subs({xs: rr * sp.cos(tt), ys: rr * sp.sin(tt)}) * rr
rhs = sp.integrate(sp.integrate(polar, (rr, 0, 1)), (tt, 0, 2 * sp.pi))
print("curl flux through the paraboloid =", sp.simplify(rhs))

flat = sp.integrate(sp.integrate(-1 * rr, (rr, 0, 1)), (tt, 0, 2 * sp.pi))   # flat disk, N = (0,0,1)
print("curl flux through the flat disk  =", sp.simplify(flat))
assert sp.simplify(lhs + sp.pi) == 0 and sp.simplify(rhs + sp.pi) == 0 and sp.simplify(flat + sp.pi) == 0

boundary circulation = -pi
curl flux through the paraboloid = -pi
curl flux through the flat disk  = -pi


All three equal $-\pi$. The last two agreeing is Theorem 4.6's corollary: the curl flux depends only on the shared boundary, not on the surface.

---

### Problem L1.9 — Gauss's Divergence Theorem Computation
**Source:** Stewart, *Multivariable Calculus*, Ch. 16.9  

**Problem Statement:**  
Use Gauss's Divergence Theorem to compute the outward flux $\oiint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S$ of field $\mathbf{F}(x,y,z) = x^3\mathbf{i} + y^3\mathbf{j} + z^3\mathbf{k}$ through sphere $S: x^2 + y^2 + z^2 = R^2$.

#### First-Principles Intuition
Gauss's Theorem converts the surface flux into a 3D volume integral of $\nabla \cdot \mathbf{F}$. In spherical coordinates, $x^2+y^2+z^2 = r^2$.

#### Step-by-Step Solution
1. Compute divergence:

$$
   \nabla \cdot \mathbf{F} = \frac{\partial}{\partial x}(x^3) + \frac{\partial}{\partial y}(y^3) + \frac{\partial}{\partial z}(z^3) = 3x^2 + 3y^2 + 3z^2 = 3(x^2 + y^2 + z^2)
$$

2. Apply Divergence Theorem:

$$
   \oiint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S = \iiint_V 3(x^2 + y^2 + z^2)\,\mathrm{d}V
$$

3. Convert to spherical coordinates ($x^2+y^2+z^2 = r^2, \mathrm{d}V = r^2\sin\phi\,\mathrm{d}r\mathrm{d}\phi\mathrm{d}\theta$):

$$
   = 3 \int_0^{2\pi}\mathrm{d}\theta \int_0^\pi \sin\phi\,\mathrm{d}\phi \int_0^R r^2 \cdot r^2\,\mathrm{d}r
$$

$$
   = 3 (2\pi) (2) \left[ \frac{r^5}{5} \right]_0^R = 12\pi \frac{R^5}{5} = \frac{12\pi R^5}{5}
$$

$$
\boxed{\frac{12\pi R^5}{5}}
$$

#### Key Insight / Takeaway
A 2D surface flux integral over a 3D sphere is simplified to a trivial 1D radial volume integral using spherical symmetry and divergence.

---

Both sides of the divergence theorem, kept symbolic in $R$.

In [14]:
# L1.9: divergence theorem for F = (x^3, y^3, z^3) on a ball of radius R.
R, r, ph, th = sp.symbols("R r phi theta", positive=True)
vol = sp.integrate(sp.integrate(sp.integrate(3 * r**2 * r**2 * sp.sin(ph),
                                             (r, 0, R)), (ph, 0, sp.pi)), (th, 0, 2 * sp.pi))
print("volume side =", sp.simplify(vol))

# Surface side: on |x| = R, F . n = R^2 (x^4+y^4+z^4)/R^4 * R^2 ... integrate directly.
surf = sp.integrate(sp.integrate(
    R**3 * ((sp.sin(ph) * sp.cos(th))**4 + (sp.sin(ph) * sp.sin(th))**4 + sp.cos(ph)**4)
    * R**2 * sp.sin(ph), (ph, 0, sp.pi)), (th, 0, 2 * sp.pi))
print("surface side =", sp.simplify(surf))
assert sp.simplify(vol - 12 * sp.pi * R**5 / 5) == 0 and sp.simplify(surf - vol) == 0

volume side = 12*pi*R**5/5


surface side = 12*pi*R**5/5


The two sides agree as symbolic expressions in $R$, so the identity holds for every radius, not just one sampled value.

---

### Problem L1.10 — Divergence & Curl Vector Identity Derivations
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 4.4  

**Problem Statement:**  
Let $f: \mathbb{R}^3 \to \mathbb{R}$ be a $C^1$ scalar field and $\mathbf{F}: \mathbb{R}^3 \to \mathbb{R}^3$ be a $C^1$ vector field. Derive product rules:  
1. $\nabla \cdot (f\mathbf{F}) = f(\nabla \cdot \mathbf{F}) + \nabla f \cdot \mathbf{F}$  
2. $\nabla \times (f\mathbf{F}) = f(\nabla \times \mathbf{F}) + (\nabla f) \times \mathbf{F}$

#### First-Principles Intuition
Apply single-variable product rules $(fP)_x = f_x P + f P_x$ componentwise to vector operators.

#### Step-by-Step Solution
1. Divergence product rule: Let $\mathbf{F} = (P, Q, R)$.

$$
   \nabla \cdot (f\mathbf{F}) = \frac{\partial}{\partial x}(fP) + \frac{\partial}{\partial y}(fQ) + \frac{\partial}{\partial z}(fR)
$$

$$
   = \left(\frac{\partial f}{\partial x}P + f\frac{\partial P}{\partial x}\right) + \left(\frac{\partial f}{\partial y}Q + f\frac{\partial Q}{\partial y}\right) + \left(\frac{\partial f}{\partial z}R + f\frac{\partial R}{\partial z}\right)
$$

   Regrouping terms:

$$
   = f\left(\frac{\partial P}{\partial x} + \frac{\partial Q}{\partial y} + \frac{\partial R}{\partial z}\right) + \left(\frac{\partial f}{\partial x}P + \frac{\partial f}{\partial y}Q + \frac{\partial f}{\partial z}R\right) = f(\nabla \cdot \mathbf{F}) + \nabla f \cdot \mathbf{F}
$$

2. Curl product rule $x$-component:

$$
   [\nabla \times (f\mathbf{F})]_x = \frac{\partial}{\partial y}(fR) - \frac{\partial}{\partial z}(fQ) = \left(f\frac{\partial R}{\partial y} + \frac{\partial f}{\partial y}R\right) - \left(f\frac{\partial Q}{\partial z} + \frac{\partial f}{\partial z}Q\right)
$$

$$
   = f\left(\frac{\partial R}{\partial y} - \frac{\partial Q}{\partial z}\right) + \left(\frac{\partial f}{\partial y}R - \frac{\partial f}{\partial z}Q\right) = f[\nabla \times \mathbf{F}]_x + [(\nabla f) \times \mathbf{F}]_x
$$

   Repeating for $y$ and $z$ components establishes the full vector identity.

$$
\boxed{\nabla \cdot (f\mathbf{F}) = f(\nabla \cdot \mathbf{F}) + \nabla f \cdot \mathbf{F}, \quad \nabla \times (f\mathbf{F}) = f(\nabla \times \mathbf{F}) + (\nabla f) \times \mathbf{F}}
$$

#### Key Insight / Takeaway
Differential vector operators satisfy product derivative expansion rules directly analogous to scalar calculus product rules.

---

Both product rules against generic undetermined functions, so the residual is an identity, not a coincidence at a sample point.

In [15]:
# L1.10: the two product rules, checked symbolically on generic C^1 data.
x, y, z = sp.symbols("x y z")
f = sp.Function("f")(x, y, z)
P, Q, R = (sp.Function(n)(x, y, z) for n in ("P", "Q", "R"))
F = sp.Matrix([P, Q, R])
V = (x, y, z)
grad = lambda g: sp.Matrix([sp.diff(g, v) for v in V])
div = lambda G: sum(sp.diff(G[i], V[i]) for i in range(3))
curl = lambda G: sp.Matrix([sp.diff(G[2], y) - sp.diff(G[1], z),
                            sp.diff(G[0], z) - sp.diff(G[2], x),
                            sp.diff(G[1], x) - sp.diff(G[0], y)])
res_div = sp.simplify(div(f * F) - (f * div(F) + grad(f).dot(F)))
res_curl = sp.simplify(curl(f * F) - (f * curl(F) + grad(f).cross(F)))
print("divergence rule residual =", res_div)
print("curl rule residual       =", res_curl.T)
assert res_div == 0 and res_curl == sp.zeros(3, 1)

divergence rule residual = 0
curl rule residual       = Matrix([[0, 0, 0]])


Both residuals are exactly zero for undetermined $f, P, Q, R$, which is the identity itself rather than a numerical near-miss.

---

## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Gauss's Law for Gravitation
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 5  

**Problem Statement:**  
The gravitational field of mass $M$ at origin is $\mathbf{g}(\mathbf{r}) = -\frac{GM}{r^2}\hat{\mathbf{r}} = -\frac{GM}{r^3}\mathbf{r}$.  
1. Show that $\nabla \cdot \mathbf{g} = 0$ for all $\mathbf{r} \neq \mathbf{0}$.  
2. Compute flux $\oiint_S \mathbf{g} \cdot \mathbf{n}\,\mathrm{d}S$ for sphere $S$ of radius $R$ centered at origin.  
3. Explain why the flux is non-zero despite $\nabla \cdot \mathbf{g} = 0$ away from origin.

#### First-Principles Intuition
The inverse-square force has zero divergence everywhere except at origin $r=0$, where a point source delta-function density resides.

#### Step-by-Step Solution
1. Compute divergence for $\mathbf{r} \neq \mathbf{0}$: $\mathbf{g} = -GM (x(x^2+y^2+z^2)^{-\frac{3}{2}}, y(\dots)^{-\frac{3}{2}}, z(\dots)^{-\frac{3}{2}})$.

$$
   \frac{\partial g_x}{\partial x} = -GM \left( (x^2+y^2+z^2)^{-\frac{3}{2}} - 3x^2(x^2+y^2+z^2)^{-\frac{5}{2}} \right) = -GM \left( \frac{1}{r^3} - \frac{3x^2}{r^5} \right)
$$

   Summing all three partial derivatives:

$$
   \nabla \cdot \mathbf{g} = -GM \left( \frac{3}{r^3} - \frac{3(x^2+y^2+z^2)}{r^5} \right) = -GM \left( \frac{3}{r^3} - \frac{3}{r^3} \right) = 0
$$

2. Compute sphere surface flux ($r=R$): On sphere, $\mathbf{n} = \hat{\mathbf{r}}$, $\mathbf{g} \cdot \mathbf{n} = -\frac{GM}{R^2}$.

$$
   \Phi = \oiint_S \left(-\frac{GM}{R^2}\right)\mathrm{d}S = -\frac{GM}{R^2} (4\pi R^2) = -4\pi G M
$$

3. Gauss's Divergence theorem requires $C^1$ continuity on the *entire* enclosed volume. The singularity at $r=0$ acts as a Dirac delta distribution $\nabla \cdot \mathbf{g} = -4\pi G M \delta^3(\mathbf{r})$.

$$
\boxed{\nabla \cdot \mathbf{g} = 0 \quad (\mathbf{r} \neq \mathbf{0}), \quad \oiint_S \mathbf{g} \cdot \mathbf{n}\,\mathrm{d}S = -4\pi G M}
$$

#### Key Insight / Takeaway
Inverse-square law fields concentrate all divergence at point singularities, producing constant non-zero flux across any enclosing surface.

---

The divergence off the origin, and the flux, symbolically in $G$, $M$ and $R$.

In [16]:
# L2.1: inverse-square field -- zero divergence off the origin, flux -4*pi*G*M through any sphere.
G, M = sp.symbols("G M", positive=True)
x, y, z = sp.symbols("x y z")
rmag = sp.sqrt(x**2 + y**2 + z**2)
g = -G * M * sp.Matrix([x, y, z]) / rmag**3
divg = sp.simplify(sum(sp.diff(g[i], v) for i, v in enumerate((x, y, z))))
print("div g (off the origin) =", divg)

Rs = sp.symbols("R", positive=True)
flux = sp.simplify(-G * M / Rs**2 * 4 * sp.pi * Rs**2)
print("flux through the sphere of radius R =", flux, "-- independent of R")
assert divg == 0 and sp.simplify(flux + 4 * sp.pi * G * M) == 0

div g (off the origin) = 0
flux through the sphere of radius R = -4*pi*G*M -- independent of R


The divergence is exactly zero off the origin, while the flux is $-4\pi GM$ regardless of $R$ — all of the divergence has collapsed into the singular point.

---

### Problem L2.2 — Faraday's Law of Induction
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 17  

**Problem Statement:**  
Faraday's law of induction in integral form states that the electromotive force $\oint_{\partial S} \mathbf{E} \cdot \mathrm{d}\mathbf{r}$ around loop $\partial S$ equals $-\frac{\mathrm{d}}{\mathrm{d}t} \iint_S \mathbf{B} \cdot \mathbf{n}\,\mathrm{d}S$. Use Stokes' Theorem to derive Maxwell's differential equation $\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}$.

#### First-Principles Intuition
Apply Stokes' Theorem to convert the boundary line integral into a surface integral of curl over an arbitrary surface $S$.

#### Step-by-Step Solution
1. Apply Stokes' Theorem to the left hand side:

$$
   \oint_{\partial S} \mathbf{E} \cdot \mathrm{d}\mathbf{r} = \iint_S (\nabla \times \mathbf{E}) \cdot \mathbf{n}\,\mathrm{d}S
$$

2. For a fixed stationary surface $S$, bring the time derivative inside the right hand integral:

$$
   -\frac{\mathrm{d}}{\mathrm{d}t} \iint_S \mathbf{B} \cdot \mathbf{n}\,\mathrm{d}S = \iint_S \left( -\frac{\partial \mathbf{B}}{\partial t} \right) \cdot \mathbf{n}\,\mathrm{d}S
$$

3. Equating both surface integrals:

$$
   \iint_S \left( \nabla \times \mathbf{E} + \frac{\partial \mathbf{B}}{\partial t} \right) \cdot \mathbf{n}\,\mathrm{d}S = 0
$$

4. Since this identity holds for *any* arbitrary surface $S$, the integrand must vanish identically everywhere.

$$
\boxed{\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}}
$$

#### Key Insight / Takeaway
Field theorems bridge global empirical integral observations to localized continuous differential equations.

---

### Problem L2.3 — Fluid Flow & 2D Stream Function
**Source:** Apostol, *Calculus, Vol. II*, Ch. 11  

**Problem Statement:**  
For a 2D incompressible ($\nabla \cdot \mathbf{v} = 0$) and irrotational ($\nabla \times \mathbf{v} = \mathbf{0}$) velocity field $\mathbf{v} = u\mathbf{i} + v\mathbf{j}$, prove that:  
1. There exists a velocity potential $\phi$ such that $u = \frac{\partial \phi}{\partial x}, v = \frac{\partial \phi}{\partial y}$.  
2. There exists a stream function $\psi$ such that $u = \frac{\partial \psi}{\partial y}, v = -\frac{\partial \psi}{\partial x}$.  
3. $\phi$ and $\psi$ satisfy the Cauchy-Riemann equations of complex analysis.

#### First-Principles Intuition
Irrotationality provides $\mathbf{v} = \nabla \phi$. Incompressibility $u_x + v_y = 0$ allows defining stream function $\psi$ via exact differentials $\mathrm{d}\psi = -v\mathrm{d}x + u\mathrm{d}y$.

#### Step-by-Step Solution
1. Irrotationality $\nabla \times \mathbf{v} = 0 \implies \frac{\partial v}{\partial x} - \frac{\partial u}{\partial y} = 0$. By potential theory, $\mathbf{v} = \nabla \phi \implies u = \frac{\partial \phi}{\partial x}, v = \frac{\partial \phi}{\partial y}$.
2. Incompressibility $\nabla \cdot \mathbf{v} = 0 \implies \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0$. By Green's flux formulation, differential $-v\mathrm{d}x + u\mathrm{d}y$ is exact. Thus there exists $\psi$ such that $\frac{\partial \psi}{\partial y} = u$ and $\frac{\partial \psi}{\partial x} = -v$.
3. Equating formulas for $u$ and $v$:

$$
   \frac{\partial \phi}{\partial x} = \frac{\partial \psi}{\partial y} \quad \text{and} \quad \frac{\partial \phi}{\partial y} = -\frac{\partial \psi}{\partial x}
$$

   These are precisely the Cauchy-Riemann equations for analytic complex function $F(z) = \phi + i\psi$.

$$
\boxed{\frac{\partial \phi}{\partial x} = \frac{\partial \psi}{\partial y}, \quad \frac{\partial \phi}{\partial y} = -\frac{\partial \psi}{\partial x} \quad (\text{Cauchy-Riemann Equations})}
$$

#### Key Insight / Takeaway
2D ideal fluid dynamics is completely equivalent to complex analytic function theory via field potential and stream operators.

---

### Problem L2.4 — Vector Potential $\mathbf{A}$ for Uniform Magnetic Field
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 14  

**Problem Statement:**  
Given uniform magnetic field $\mathbf{B} = B_0\mathbf{k}$:  
1. Find a symmetric vector potential $\mathbf{A}(x,y,z)$ such that $\nabla \times \mathbf{A} = \mathbf{B}$ and $\nabla \cdot \mathbf{A} = 0$ (Coulomb gauge).  
2. Verify $\oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r} = \iint_S \mathbf{B} \cdot \mathbf{n}\,\mathrm{d}S$ for circle of radius $R$ in $xy$-plane.

#### First-Principles Intuition
A rotational magnetic field can be generated by azimuthal circulating potential $\mathbf{A} = \frac{1}{2}(\mathbf{B} \times \mathbf{r})$.

#### Step-by-Step Solution
1. Try symmetric field $\mathbf{A} = -\frac{B_0 y}{2}\mathbf{i} + \frac{B_0 x}{2}\mathbf{j} + 0\mathbf{k}$.
   - Compute divergence: $\nabla \cdot \mathbf{A} = \frac{\partial}{\partial x}(-\frac{B_0 y}{2}) + \frac{\partial}{\partial y}(\frac{B_0 x}{2}) = 0$.
   - Compute curl:

$$
     \nabla \times \mathbf{A} = \left( \frac{\partial}{\partial x}\left(\frac{B_0 x}{2}\right) - \frac{\partial}{\partial y}\left(-\frac{B_0 y}{2}\right) \right)\mathbf{k} = \left(\frac{B_0}{2} + \frac{B_0}{2}\right)\mathbf{k} = B_0\mathbf{k} = \mathbf{B}
$$

2. Evaluate boundary line integral along circle $\mathbf{r}(\theta) = (R\cos\theta, R\sin\theta, 0)$:
   - $\mathbf{A} = \frac{B_0 R}{2}(-\sin\theta \mathbf{i} + \cos\theta \mathbf{j})$.
   - $\mathrm{d}\mathbf{r} = R(-\sin\theta \mathbf{i} + \cos\theta \mathbf{j})\mathrm{d}\theta$.
   - $\oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r} = \frac{B_0 R^2}{2} \int_0^{2\pi} (\sin^2\theta + \cos^2\theta)\mathrm{d}\theta = \frac{B_0 R^2}{2}(2\pi) = B_0 (\pi R^2)$.
3. Surface magnetic flux: $\iint_S \mathbf{B} \cdot \mathbf{n}\,\mathrm{d}S = B_0 \cdot \text{Area}(\text{disk}) = B_0 (\pi R^2)$.

$$
\boxed{\mathbf{A} = \frac{B_0}{2}(-y\mathbf{i} + x\mathbf{j}), \quad \text{Flux} = \pi R^2 B_0}
$$

#### Key Insight / Takeaway
Vector potential circulation measures total magnetic flux enclosed by the boundary curve.

---

The gauge conditions and the circulation-equals-flux identity, symbolically.

In [17]:
# L2.4: the symmetric gauge A = (B0/2)(-y, x, 0) reproduces B = B0 k, and its circulation is the flux.
x, y, z, B0, Rr = sp.symbols("x y z B_0 R", positive=True)
A = sp.Matrix([-B0 * y / 2, B0 * x / 2, 0])
curlA = sp.Matrix([sp.diff(A[2], y) - sp.diff(A[1], z),
                   sp.diff(A[0], z) - sp.diff(A[2], x),
                   sp.diff(A[1], x) - sp.diff(A[0], y)])
divA = sp.simplify(sum(sp.diff(A[i], v) for i, v in enumerate((x, y, z))))
print("curl A =", curlA.T, "  div A =", divA)

th = sp.symbols("theta")
c = sp.Matrix([Rr * sp.cos(th), Rr * sp.sin(th), 0])
circ = sp.simplify(sp.integrate(A.subs({x: c[0], y: c[1], z: c[2]}).dot(sp.diff(c, th)), (th, 0, 2 * sp.pi)))
print("circulation of A =", circ, "  flux of B =", sp.pi * Rr**2 * B0)
assert list(curlA) == [0, 0, B0] and divA == 0 and sp.simplify(circ - sp.pi * Rr**2 * B0) == 0

curl A = Matrix([[0, 0, B_0]])   div A = 0


circulation of A = pi*B_0*R**2   flux of B = pi*B_0*R**2


$\nabla \times \mathbf{A} = B_0\mathbf{k}$, $\nabla \cdot \mathbf{A} = 0$, and the circulation equals $\pi R^2 B_0$, the enclosed flux.

---

### Problem L2.5 — Heat Equation & Thermal Flux
**Source:** Marsden & Tromba, *Vector Calculus*, Ch. 7.3  

**Problem Statement:**  
Fourier's Law of heat conduction states that heat flux vector is $\mathbf{q} = -k \nabla T$, where $k \gt 0$ is thermal conductivity and $T$ is temperature.  
Use Gauss's Divergence Theorem on thermal energy balance $\frac{\mathrm{d}}{\mathrm{d}t}\iiint_V c\rho T\,\mathrm{d}V = -\oiint_{\partial V} \mathbf{q} \cdot \mathbf{n}\,\mathrm{d}S$ to derive the Heat Diffusion Equation $\frac{\partial T}{\partial t} = \alpha \nabla^2 T$ (where thermal diffusivity $\alpha = \frac{k}{c\rho}$).

#### First-Principles Intuition
Heat energy change inside volume $V$ must equal net heat flowing inward through boundary surface $\partial V$.

#### Step-by-Step Solution
1. Apply Divergence Theorem to surface heat flux integral:

$$
   \oiint_{\partial V} \mathbf{q} \cdot \mathbf{n}\,\mathrm{d}S = \iiint_V (\nabla \cdot \mathbf{q})\,\mathrm{d}V = \iiint_V \nabla \cdot (-k\nabla T)\,\mathrm{d}V = -k \iiint_V \nabla^2 T\,\mathrm{d}V
$$

2. Substitute into energy conservation relation:

$$
   \iiint_V c\rho \frac{\partial T}{\partial t}\,\mathrm{d}V = k \iiint_V \nabla^2 T\,\mathrm{d}V
$$

3. Rearrange into a single volume integral:

$$
   \iiint_V \left( c\rho \frac{\partial T}{\partial t} - k \nabla^2 T \right)\mathrm{d}V = 0
$$

4. Holding for arbitrary volume $V$ forces the integrand to zero: $c\rho \frac{\partial T}{\partial t} = k \nabla^2 T \implies \frac{\partial T}{\partial t} = \alpha \nabla^2 T$.

$$
\boxed{\frac{\partial T}{\partial t} = \alpha \nabla^2 T \quad \text{where } \alpha = \frac{k}{c\rho}}
$$

#### Key Insight / Takeaway
The Laplacian operator $\nabla^2 T = \nabla \cdot (\nabla T)$ physically represents local net thermal divergence/inflow across surrounding spatial neighborhoods.

---

### Problem L2.6 — Solid Angle Flux Integral
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 4380  

**Problem Statement:**  
Prove that the solid angle $\Omega$ subtended by an oriented smooth surface $S$ from the origin is given by the vector surface integral:

$$
\Omega = \iint_S \frac{\mathbf{r} \cdot \mathbf{n}}{r^3}\,\mathrm{d}S
$$

#### First-Principles Intuition
Solid angle measures the projected area of surface $S$ onto the unit sphere centered at origin. The radial vector field $\mathbf{F} = \frac{\mathbf{r}}{r^3} = \frac{\hat{\mathbf{r}}}{r^2}$ projects surface area patches radially onto unit sphere segments.

#### Step-by-Step Solution
1. Project surface patch $\mathrm{d}S$ onto sphere of radius $r$: projection area orthogonal to radius vector is $\cos\theta\mathrm{d}S = (\hat{\mathbf{r}} \cdot \mathbf{n})\mathrm{d}S$.
2. By definition, solid angle element $\mathrm{d}\Omega$ is the projected area divided by $r^2$:

$$
   \mathrm{d}\Omega = \frac{(\hat{\mathbf{r}} \cdot \mathbf{n})\mathrm{d}S}{r^2} = \frac{\mathbf{r} \cdot \mathbf{n}}{r^3}\,\mathrm{d}S
$$

3. Integrating over the entire surface $S$:

$$
   \Omega = \iint_S \frac{\mathbf{r} \cdot \mathbf{n}}{r^3}\,\mathrm{d}S
$$

$$
\boxed{\Omega = \iint_S \frac{\mathbf{r} \cdot \mathbf{n}}{r^3}\,\mathrm{d}S}
$$

#### Key Insight / Takeaway
The inverse-square flux integral geometrically computes 3D solid angle subtended by arbitrary spatial surfaces.

---

### Problem L2.7 — Continuous Normalizing Flows & Density Evolution
**Source:** Chen et al. (2018), *Neural Ordinary Differential Equations*  

**Problem Statement:**  
In continuous normalizing flows, sample state $\mathbf{z}(t) \in \mathbb{R}^d$ evolves via Neural ODE velocity field $\frac{\mathrm{d}\mathbf{z}}{\mathrm{d}t} = \mathbf{f}(\mathbf{z}(t), t)$.  
Using the continuity equation $\frac{\partial p_t}{\partial t} + \nabla \cdot (p_t \mathbf{f}) = 0$, prove that the log-density of state $\mathbf{z}(t)$ evolves as:

$$
\frac{\mathrm{d}}{\mathrm{d}t} \log p_t(\mathbf{z}(t)) = -\nabla \cdot \mathbf{f}(\mathbf{z}(t), t)
$$

#### First-Principles Intuition
By following a sample along its trajectory, the total material derivative $\frac{\mathrm{d}}{\mathrm{d}t}$ accounts for both local time changes and spatial velocity drift. Local volume expansion of velocity field ($\nabla \cdot \mathbf{f}$) dilutes probability density.

#### Step-by-Step Solution
1. Expand total time derivative of $p_t(\mathbf{z}(t))$ via multivariable chain rule:

$$
   \frac{\mathrm{d}}{\mathrm{d}t} p_t(\mathbf{z}(t)) = \frac{\partial p_t}{\partial t} + \nabla p_t \cdot \frac{\mathrm{d}\mathbf{z}}{\mathrm{d}t} = \frac{\partial p_t}{\partial t} + \nabla p_t \cdot \mathbf{f}
$$

2. Substitute Continuity Equation $\frac{\partial p_t}{\partial t} = -\nabla \cdot (p_t \mathbf{f})$:

$$
   \frac{\mathrm{d}}{\mathrm{d}t} p_t(\mathbf{z}(t)) = -\nabla \cdot (p_t \mathbf{f}) + \nabla p_t \cdot \mathbf{f}
$$

3. Expand vector product rule $\nabla \cdot (p_t \mathbf{f}) = p_t (\nabla \cdot \mathbf{f}) + \nabla p_t \cdot \mathbf{f}$:

$$
   \frac{\mathrm{d}}{\mathrm{d}t} p_t(\mathbf{z}(t)) = -\Big( p_t (\nabla \cdot \mathbf{f}) + \nabla p_t \cdot \mathbf{f} \Big) + \nabla p_t \cdot \mathbf{f} = -p_t (\nabla \cdot \mathbf{f})
$$

4. Divide by $p_t(\mathbf{z}(t))$:

$$
   \frac{1}{p_t} \frac{\mathrm{d}p_t}{\mathrm{d}t} = -\nabla \cdot \mathbf{f} \implies \frac{\mathrm{d}}{\mathrm{d}t} \log p_t(\mathbf{z}(t)) = -\nabla \cdot \mathbf{f}(\mathbf{z}(t), t)
$$

$$
\boxed{\frac{\mathrm{d}}{\mathrm{d}t} \log p_t(\mathbf{z}(t)) = -\nabla \cdot \mathbf{f}(\mathbf{z}(t), t)}
$$

#### Key Insight / Takeaway
Evaluating high-dimensional density changes in Neural ODEs requires computing only the scalar divergence of the neural velocity field.

---

### Problem L2.8 — Helmholtz Decomposition of GAN Training Field
**Source:** Mescheder et al., *Which Training Methods for GANs do actually Converge?*  

**Problem Statement:**  
Consider a zero-sum game with parameters $(x, y) \in \mathbb{R}^2$ and minimax loss $L(x, y) = xy$. Gradient descent-ascent updates follow velocity field $\mathbf{V}(x, y) = (-\nabla_x L, \nabla_y L)^T = (-y, x)^T$.  
1. Show that $\nabla \cdot \mathbf{V} = 0$ and $\nabla \times \mathbf{V} = 2\mathbf{k}$.  
2. Decompose $\mathbf{V}$ using Helmholtz decomposition into potential field $-\nabla \phi$ and solenoidal field $\mathbf{V}_{\text{rot}}$.  
3. Explain why standard gradient updates orbit in closed circles instead of reaching equilibrium $(0,0)$.

#### First-Principles Intuition
Pure gradient descent on scalar loss yields irrotational potential flow. Multi-agent games introduce non-zero curl, creating rotational solenoidal fields that cause infinite non-convergent rotations around equilibrium.

#### Step-by-Step Solution
1. Compute divergence and curl:
   - $\nabla \cdot \mathbf{V} = \frac{\partial}{\partial x}(-y) + \frac{\partial}{\partial y}(x) = 0$.
   - $\nabla \times \mathbf{V} = \left(\frac{\partial}{\partial x}(x) - \frac{\partial}{\partial y}(-y)\right)\mathbf{k} = 2\mathbf{k}$.
2. Helmholtz decomposition: Since divergence is zero, potential $\phi = 0$. The entire vector field is purely solenoidal/rotational: $\mathbf{V} = \mathbf{V}_{\text{rot}} = (-y, x)^T$.
3. Trajectory differential equations:

$$
   \frac{\mathrm{d}x}{\mathrm{d}t} = -y, \quad \frac{\mathrm{d}y}{\mathrm{d}t} = x
$$

   Taking second derivative: $\frac{\mathrm{d}^2 x}{\mathrm{d}t^2} = -\frac{\mathrm{d}y}{\mathrm{d}t} = -x \implies x(t) = R\cos(t + \delta), y(t) = R\sin(t + \delta)$.
   The state vector orbits on concentric circles $x^2 + y^2 = R^2$ with constant distance from origin!

$$
\boxed{\nabla \cdot \mathbf{V} = 0, \quad \nabla \times \mathbf{V} = 2\mathbf{k}, \quad \mathbf{V} = \mathbf{V}_{\text{rot}} \quad (\text{Pure Rotational Flow})}
$$

#### Key Insight / Takeaway
GAN training instabilities arise directly from non-zero vector field curl; convergence requires symplectic damping techniques to suppress solenoidal dynamics.

---

The two field operators, and the radius of the exact orbit along four full turns.

In [18]:
# L2.8: the GAN field V = (-y, x) is divergence free with curl 2k, and its orbits are circles.
x, y = sp.symbols("x y")
V = sp.Matrix([-y, x, 0])
print("div V =", sp.diff(V[0], x) + sp.diff(V[1], y),
      "  curl V =", sp.Matrix([0, 0, sp.diff(V[1], x) - sp.diff(V[0], y)]).T)

# Integrate the flow numerically for four full turns and watch the radius.
sol = integrate.solve_ivp(lambda t, s: [-s[1], s[0]], (0, 4 * np.pi), [1.0, 0.0],
                          rtol=1e-11, atol=1e-12, dense_output=True)
ts = np.linspace(0, 4 * np.pi, 400)
radii = np.linalg.norm(sol.sol(ts), axis=0)
print(f"orbit radius over four turns: min {radii.min():.10f}, max {radii.max():.10f}")
print(f"endpoint {sol.sol(4 * np.pi)} vs start [1. 0.] -- the orbit closed, it did not converge")
assert sp.diff(V[0], x) + sp.diff(V[1], y) == 0
assert sp.diff(V[1], x) - sp.diff(V[0], y) == 2
assert np.ptp(radii) < 1e-8

div V = 0   curl V = Matrix([[0, 0, 2]])


orbit radius over four turns: min 1.0000000000, max 1.0000000000
endpoint [1. 0.] vs start [1. 0.] -- the orbit closed, it did not converge


Divergence $0$, curl $2$, and the orbit radius constant to twelve digits: the flow neither contracts nor expands phase-space area, so gradient descent-ascent can only circle the equilibrium.

---

### Problem L2.9 — Physics-Informed Neural Network (PINN) Conservation Loss
**Source:** Raissi et al. (2019), *Physics-Informed Neural Networks*  

**Problem Statement:**  
A PINN outputs predicted 2D fluid velocity $\mathbf{u}_\theta(x, y) = (u_\theta(x,y), v_\theta(x,y))^T$. To enforce mass conservation for an incompressible fluid ($\nabla \cdot \mathbf{u} = 0$), we define loss term $L_{\operatorname{div}}(\theta) = \frac{1}{N}\sum_{i=1}^N \lvert \nabla \cdot \mathbf{u}_\theta(x_i, y_i) \rvert^2$.  
Write down explicit partial derivatives required by automatic differentiation to evaluate $L_{\operatorname{div}}$.

#### First-Principles Intuition
Vector calculus operators provide exact differential residual constraints used as loss penalties in physics-informed AI models.

#### Step-by-Step Solution
1. Divergence of predicted velocity field:

$$
   \nabla \cdot \mathbf{u}_\theta(x, y) = \frac{\partial u_\theta}{\partial x}(x, y) + \frac{\partial v_\theta}{\partial y}(x, y)
$$

2. Squared divergence residual at collocation point $(x_i, y_i)$:

$$
   r_i(\theta) = \frac{\partial u_\theta}{\partial x}(x_i, y_i) + \frac{\partial v_\theta}{\partial y}(x_i, y_i)
$$

3. Physics residual loss function:

$$
   L_{\operatorname{div}}(\theta) = \frac{1}{N}\sum_{i=1}^N \left( \frac{\partial u_\theta}{\partial x}(x_i, y_i) + \frac{\partial v_\theta}{\partial y}(x_i, y_i) \right)^2
$$

$$
\boxed{L_{\operatorname{div}}(\theta) = \frac{1}{N}\sum_{i=1}^N \left( \frac{\partial u_\theta}{\partial x} + \frac{\partial v_\theta}{\partial y} \right)^2}
$$

#### Key Insight / Takeaway
Divergence operators evaluated via automatic differentiation serve as exact spatial inductive biases in neural field solvers.

---

### Problem L2.10 — Work Done by Magnetic Lorentz Force
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 13  

**Problem Statement:**  
Prove that the magnetic Lorentz force $\mathbf{F}_m = q(\mathbf{v} \times \mathbf{B})$ does zero mechanical work on a charged particle moving along any trajectory $C$.

#### First-Principles Intuition
Work is $\int_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_{t_1}^{t_2} (\mathbf{F} \cdot \mathbf{v})\mathrm{d}t$. The magnetic force is always perpendicular to velocity vector $\mathbf{v}$.

#### Step-by-Step Solution
1. Instantaneous power delivered by force $\mathbf{F}_m$:

$$
   P = \mathbf{F}_m \cdot \mathbf{v} = q(\mathbf{v} \times \mathbf{B}) \cdot \mathbf{v}
$$

2. By standard triple scalar product vector identity:

$$
   (\mathbf{v} \times \mathbf{B}) \cdot \mathbf{v} = \mathbf{B} \cdot (\mathbf{v} \times \mathbf{v}) = \mathbf{B} \cdot \mathbf{0} = 0
$$

3. Work integrated along trajectory $C$:

$$
   W = \int_C \mathbf{F}_m \cdot \mathrm{d}\mathbf{r} = \int_{t_1}^{t_2} (\mathbf{F}_m \cdot \mathbf{v})\,\mathrm{d}t = \int_{t_1}^{t_2} 0\,\mathrm{d}t = 0
$$

$$
\boxed{W = 0}
$$

#### Key Insight / Takeaway
Magnetic fields alter particle direction without changing kinetic energy or doing mechanical work.

---

The zero-power claim across 2000 random velocity and field pairs, reported relative to the natural scale $q\lVert \mathbf{v} \rVert^2 \lVert \mathbf{B} \rVert$.

In [19]:
# L2.10: the magnetic force is orthogonal to the velocity, so its power vanishes -- 2000 random samples.
q = 1.6e-19
v = rng.normal(size=(2000, 3))
B = rng.normal(size=(2000, 3))
power = np.einsum("ij,ij->i", q * np.cross(v, B), v)
scale = q * np.linalg.norm(v, axis=1) ** 2 * np.linalg.norm(B, axis=1)
print(f"max |power| = {np.abs(power).max():.3e}")
print(f"max relative power = {np.abs(power / scale).max():.3e}  (machine-epsilon level)")
assert np.abs(power / scale).max() < 1e-14

max |power| = 2.246e-34
max relative power = 1.726e-16  (machine-epsilon level)


The relative power sits at machine-epsilon level for every sample, so $W = 0$ is exact rather than approximate.

---

### Problem L2.11 — Hydrodynamic Circulation & Kelvin's Circulation Theorem
**Source:** Apostol, *Calculus, Vol. II*, Ch. 11  

**Problem Statement:**  
Kelvin's circulation theorem states that for an inviscid, barotropic fluid with conservative body forces ($\mathbf{f} = -\nabla \Omega$), hydrodynamic circulation $\Gamma(t) = \oint_{C(t)} \mathbf{v} \cdot \mathrm{d}\mathbf{r}$ along a material closed loop $C(t)$ moving with the fluid is constant in time: $\frac{\mathrm{d}\Gamma}{\mathrm{d}t} = 0$. Prove this result.

#### First-Principles Intuition
Differentiate under the line integral sign using material derivatives and Euler's momentum equation $\frac{\mathrm{D}\mathbf{v}}{\mathrm{D}t} = -\frac{1}{\rho}\nabla P - \nabla \Omega$.

#### Step-by-Step Solution
1. Material rate of change of circulation:

$$
   \frac{\mathrm{d}\Gamma}{\mathrm{d}t} = \frac{\mathrm{d}}{\mathrm{d}t} \oint_{C(t)} \mathbf{v} \cdot \mathrm{d}\mathbf{r} = \oint_{C(t)} \frac{\mathrm{D}\mathbf{v}}{\mathrm{D}t} \cdot \mathrm{d}\mathbf{r} + \oint_{C(t)} \mathbf{v} \cdot \mathrm{d}(\mathbf{v})
$$

2. Note $\mathbf{v} \cdot \mathrm{d}\mathbf{v} = \mathrm{d}\left(\frac{1}{2}\lVert \mathbf{v} \rVert^2\right)$. Line integral of exact gradient around closed curve is zero: $\oint_{C(t)} \mathrm{d}\left(\frac{1}{2}\lVert \mathbf{v} \rVert^2\right) = 0$.
3. Substitute Euler's equation for $\frac{\mathrm{D}\mathbf{v}}{\mathrm{D}t}$:

$$
   \frac{\mathrm{d}\Gamma}{\mathrm{d}t} = \oint_{C(t)} \left( -\frac{1}{\rho}\nabla P - \nabla \Omega \right) \cdot \mathrm{d}\mathbf{r}
$$

4. For a barotropic fluid, $\rho = \rho(P)$, so $\frac{1}{\rho}\nabla P = \nabla \int \frac{\mathrm{d}P}{\rho(P)}$.
5. Both terms are exact gradients:

$$
   \frac{\mathrm{d}\Gamma}{\mathrm{d}t} = -\oint_{C(t)} \nabla \left( \int \frac{\mathrm{d}P}{\rho} + \Omega \right) \cdot \mathrm{d}\mathbf{r} = 0
$$

$$
\boxed{\frac{\mathrm{d}\Gamma}{\mathrm{d}t} = 0 \quad (\text{Kelvin's Circulation Theorem})}
$$

#### Key Insight / Takeaway
Fluid vorticity cannot be spontaneously generated or destroyed in inviscid barotropic fluids; circulation loops are topologically preserved.

---

### Problem L2.12 — Divergence of Probability Flux in Fokker-Planck Equation
**Source:** Risken, *The Fokker-Planck Equation*  

**Problem Statement:**  
For stochastic Langevin dynamics $\mathrm{d}\mathbf{X}_t = \mathbf{b}(\mathbf{X}_t)\mathrm{d}t + \sqrt{2D}\mathrm{d}\mathbf{W}_t$, probability density $p(\mathbf{x}, t)$ obeys the Fokker-Planck continuity equation $\frac{\partial p}{\partial t} + \nabla \cdot \mathbf{J} = 0$, where probability flux is $\mathbf{J}(\mathbf{x}, t) = \mathbf{b}(\mathbf{x})p(\mathbf{x}, t) - D\nabla p(\mathbf{x}, t)$.  
Find the steady-state probability density $p_{ss}(\mathbf{x})$ when drift field is conservative $\mathbf{b}(\mathbf{x}) = -\nabla V(\mathbf{x})$ and probability flux vanishes everywhere ($\mathbf{J} = \mathbf{0}$).

#### First-Principles Intuition
Setting probability flux to zero equates drift convection to diffusive flux, generating the classic Boltzmann distribution.

#### Step-by-Step Solution
1. Set flux $\mathbf{J} = \mathbf{0}$:

$$
   \mathbf{b}(\mathbf{x})p_{ss}(\mathbf{x}) - D\nabla p_{ss}(\mathbf{x}) = \mathbf{0}
$$

2. Substitute conservative drift $\mathbf{b}(\mathbf{x}) = -\nabla V(\mathbf{x})$:

$$
   -\nabla V(\mathbf{x}) p_{ss}(\mathbf{x}) = D \nabla p_{ss}(\mathbf{x})
$$

3. Divide by $D p_{ss}(\mathbf{x})$:

$$
   \frac{\nabla p_{ss}(\mathbf{x})}{p_{ss}(\mathbf{x})} = -\frac{\nabla V(\mathbf{x})}{D} \implies \nabla \big( \log p_{ss}(\mathbf{x}) \big) = \nabla \left( -\frac{V(\mathbf{x})}{D} \right)
$$

4. Integrate potential equation:

$$
   \log p_{ss}(\mathbf{x}) = -\frac{V(\mathbf{x})}{D} + C \implies p_{ss}(\mathbf{x}) = \frac{1}{Z}\exp\left(-\frac{V(\mathbf{x})}{D}\right)
$$

   where normalization constant $Z = \int_{\mathbb{R}^d} \exp\left(-\frac{V(\mathbf{x})}{D}\right)\mathrm{d}\mathbf{x}$.

$$
\boxed{p_{ss}(\mathbf{x}) = \frac{1}{Z}\exp\left(-\frac{V(\mathbf{x})}{D}\right) \quad (\text{Boltzmann Equilibrium Distribution})}
$$

#### Key Insight / Takeaway
Zero probability flux in vector field transport yields exact analytical steady-state Gibbs-Boltzmann probability densities.

---

The Boltzmann density substituted back into $\mathbf{J} = \mathbf{b}p - D\nabla p$ for a non-quadratic potential, plus a normalisation check.

In [20]:
# L2.12: the Boltzmann density really does make the Fokker-Planck flux vanish.
x, y, D = sp.symbols("x y D", positive=True)
Vpot = (x**2 + 3 * y**2) / 2 + sp.Rational(1, 4) * x**2 * y**2
p = sp.exp(-Vpot / D)                        # unnormalised: Z is a constant and drops out
b = -sp.Matrix([sp.diff(Vpot, x), sp.diff(Vpot, y)])
J = sp.simplify(b * p - D * sp.Matrix([sp.diff(p, x), sp.diff(p, y)]))
print("probability flux J =", J.T)
assert J == sp.zeros(2, 1)

# and a 1D normalisation check for V = x^2/2, D = 1: Z = sqrt(2*pi).
Z, _ = integrate.quad(lambda u: np.exp(-u**2 / 2), -np.inf, np.inf)
print(f"Z = {Z:.12f}   sqrt(2*pi) = {np.sqrt(2*np.pi):.12f}")
assert rel(Z, np.sqrt(2 * np.pi)) < 1e-10

probability flux J = Matrix([[0, 0]])
Z = 2.506628274631   sqrt(2*pi) = 2.506628274631


The flux vector is symbolically zero for a potential that is not quadratic, so the Boltzmann form is not an artefact of the Gaussian special case.

---

## L3 — Challenge Proofs

### Problem L3.1 — Curl Flux over a Spherical Cap
**Source:** Cambridge University Mathematical Tripos, Part IA 2018  

**Problem Statement:**  
Let $S$ be the surface formed by the intersection of cylinder $x^2 + y^2 \le 1$ and upper hemisphere $x^2 + y^2 + z^2 = 2$ for $z \ge 0$.  
Compute surface integral $\iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S$ for vector field $\mathbf{F}(x,y,z) = (y^3, -x^3, z^3)$ oriented with upward normal.

#### First-Principles Intuition
By Stokes' Theorem, the surface flux of curl over hemisphere cap $S$ equals line integral around boundary curve $C$ ($x^2+y^2=1, z=1$).

#### Step-by-Step Solution
1. Identify boundary curve $C$: Intersection of $x^2+y^2=1$ and $x^2+y^2+z^2=2 \implies z^2 = 1 \implies z = 1$.
   Boundary $C$ is circle $x^2+y^2=1$ at height $z=1$.
2. Parameterize $C$: $\mathbf{r}(t) = (\cos t, \sin t, 1)$ for $t \in [0, 2\pi]$, $\mathbf{r}'(t) = (-\sin t, \cos t, 0)$.
3. Field on boundary: $\mathbf{F}(\mathbf{r}(t)) = \sin^3 t \mathbf{i} - \cos^3 t \mathbf{j} + 1 \mathbf{k}$.
4. Apply Stokes' Theorem:

$$
   \iint_S (\nabla \times \mathbf{F}) \cdot \mathbf{n}\,\mathrm{d}S = \oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \int_0^{2\pi} \Big( (\sin^3 t)(-\sin t) + (-\cos^3 t)(\cos t) + (1)(0) \Big)\mathrm{d}t
$$

$$
   = -\int_0^{2\pi} (\sin^4 t + \cos^4 t)\,\mathrm{d}t
$$

5. Integrate trigonometric identities:
   - $\sin^4 t + \cos^4 t = (\sin^2 t + \cos^2 t)^2 - 2\sin^2 t \cos^2 t = 1 - \frac{1}{2}\sin^2(2t) = 1 - \frac{1-\cos(4t)}{4} = \frac{3}{4} + \frac{\cos(4t)}{4}$.
   - $\int_0^{2\pi} \left(\frac{3}{4} + \frac{\cos(4t)}{4}\right)\mathrm{d}t = \frac{3}{4}(2\pi) = \frac{3\pi}{2}$.
6. Thus integral $= -\frac{3\pi}{2}$.

$$
\boxed{-\frac{3\pi}{2}}
$$

#### Key Insight / Takeaway
Replacing complex spherical cap integration with boundary loop parameterization reduces difficult 2D surface integrals to simple 1D trigonometric integrals.

---

The Stokes route and the direct surface integral, which must agree.

In [21]:
# L3.1: curl flux over the spherical cap, by Stokes on its boundary circle, and directly.
t = sp.symbols("t")
r = sp.Matrix([sp.cos(t), sp.sin(t), 1])
F = sp.Matrix([r[1]**3, -r[0]**3, r[2]**3])
loop = sp.simplify(sp.integrate(F.dot(sp.diff(r, t)), (t, 0, 2 * sp.pi)))
print("boundary line integral =", loop, f"= {float(loop):.12f}")

# Direct surface integral: curl F = (0, 0, -3x^2 - 3y^2) on z = sqrt(2 - x^2 - y^2),
# with upward element N = (-z_x, -z_y, 1); only the third component survives.
rr, th = sp.symbols("r theta", positive=True)
direct = sp.integrate(sp.integrate(-3 * rr**2 * rr, (rr, 0, 1)), (th, 0, 2 * sp.pi))
print("direct surface integral =", sp.simplify(direct))
assert sp.simplify(loop + 3 * sp.pi / 2) == 0 and sp.simplify(direct - loop) == 0

boundary line integral = -3*pi/2 = -4.712388980385
direct surface integral = -3*pi/2


Both routes give $-3\pi/2$. The $z^3$ component contributes nothing to the boundary integral because $\mathrm{d}z = 0$ along the circle at height $z = 1$.

---

### Problem L3.2 — Path Integral Bounds via Green's Theorem
**Source:** Immediate corollary of Green's Theorem (no verified competition attribution).

**Problem Statement:**  
Let $P, Q: \mathbb{R}^2 \to \mathbb{R}$ be continuously differentiable functions such that $\left\vert\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right\vert \le M$ everywhere on planar domain $D$ enclosed by simple curve $C$. Prove that:

$$
\left\vert \oint_C (P\,\mathrm{d}x + Q\,\mathrm{d}y) \right\vert \le M \cdot \text{Area}(D)
$$

#### First-Principles Intuition
Apply Green's Theorem and bound the double integral using the triangle inequality for integrals.

#### Step-by-Step Solution
1. By Green's Theorem:

$$
   \oint_C (P\,\mathrm{d}x + Q\,\mathrm{d}y) = \iint_D \left( \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} \right) \mathrm{d}A
$$

2. Take absolute value of both sides:

$$
   \left\vert \oint_C (P\,\mathrm{d}x + Q\,\mathrm{d}y) \right\vert = \left\vert \iint_D \left( \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} \right) \mathrm{d}A \right\vert
$$

3. Apply integral triangle inequality $\left\vert\iint_D g\mathrm{d}A\right\vert \le \iint_D \lvert g \rvert\mathrm{d}A$:

$$
   \le \iint_D \left\vert \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} \right\vert \mathrm{d}A
$$

4. Substitute bound $\left\vert\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right\vert \le M$:

$$
   \le \iint_D M\,\mathrm{d}A = M \iint_D \mathrm{d}A = M \cdot \text{Area}(D)
$$

$$
\boxed{\left\vert \oint_C (P\,\mathrm{d}x + Q\,\mathrm{d}y) \right\vert \le M \cdot \text{Area}(D)}
$$

#### Key Insight / Takeaway
Green's Theorem provides a direct geometric bridge between maximum local field vorticity $M$ and global path integral bounds.

---

### Problem L3.3 — Exterior Derivative Commutes with Pullback
**Source:** Spivak, *Calculus on Manifolds*, Ch. 4  

**Problem Statement:**  
Let $f: M \to N$ be a smooth map between manifolds, and let $\omega$ be a $k$-form on $N$. Prove the fundamental exterior algebra identity commuting pullback $f^\ast$ and exterior derivative $\mathrm{d}$:

$$
\mathrm{d}(f^\ast \omega) = f^\ast(\mathrm{d}\omega)
$$

for a 0-form $\omega = g$.

#### First-Principles Intuition
A 0-form is a scalar function $g$. Pullback $f^\ast g = g \circ f$. We must verify that exterior derivative $\mathrm{d}$ commutes with composition.

#### Step-by-Step Solution
1. Let $g: N \to \mathbb{R}$ be a 0-form on $N$. By definition of pullback:

$$
   f^\ast g = g \circ f
$$

2. Apply exterior derivative $\mathrm{d}$ to $f^\ast g$:

$$
   \mathrm{d}(f^\ast g) = \mathrm{d}(g \circ f) = \sum_{j} \frac{\partial (g \circ f)}{\partial x_j} \mathrm{d}x_j
$$

3. By multivariable chain rule, $\frac{\partial (g \circ f)}{\partial x_j} = \sum_k \left(\frac{\partial g}{\partial y_k} \circ f\right) \frac{\partial f_k}{\partial x_j}$. Thus:

$$
   \mathrm{d}(f^\ast g) = \sum_j \sum_k \left(\frac{\partial g}{\partial y_k} \circ f\right) \frac{\partial f_k}{\partial x_j} \mathrm{d}x_j = \sum_k \left(\frac{\partial g}{\partial y_k} \circ f\right) \mathrm{d}f_k
$$

4. On the other hand, compute $f^\ast(\mathrm{d}g)$:
   - $\mathrm{d}g = \sum_k \frac{\partial g}{\partial y_k} \mathrm{d}y_k$.
   - Pullback $f^\ast(\mathrm{d}g) = \sum_k f^\ast\left(\frac{\partial g}{\partial y_k}\right) f^\ast(\mathrm{d}y_k) = \sum_k \left(\frac{\partial g}{\partial y_k} \circ f\right) \mathrm{d}f_k$.
5. Both sides are identical.

$$
\boxed{\mathrm{d}(f^\ast \omega) = f^\ast(\mathrm{d}\omega)}
$$

#### Key Insight / Takeaway
The exterior derivative is a coordinate-free natural operator that commutes seamlessly with smooth manifold mappings.

---

### Problem L3.4 — Radial Flux in $n$ Dimensions
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 4395  

**Problem Statement:**  
Calculate the outward flux of position vector field $\mathbf{F}(\mathbf{x}) = \mathbf{x} = (x_1, x_2, \dots, x_n)^T$ through the boundary sphere $\partial B^n$ of $n$-dimensional ball $B^n = \{\mathbf{x} \in \mathbb{R}^n: \lVert \mathbf{x} \rVert \le R\}$, verifying the $n$-dimensional Divergence Theorem.

#### First-Principles Intuition
Compute divergence $\nabla \cdot \mathbf{x}$ in $n$-dimensions and multiply by hypervolume $V_n(R)$.

#### Step-by-Step Solution
1. Compute $n$-dimensional divergence:

$$
   \nabla \cdot \mathbf{x} = \sum_{i=1}^n \frac{\partial x_i}{\partial x_i} = \sum_{i=1}^n 1 = n
$$

2. Apply $n$-dimensional Divergence Theorem:

$$
   \Phi = \int_{\partial B^n} \mathbf{x} \cdot \mathbf{n}\,\mathrm{d}S = \int_{B^n} (\nabla \cdot \mathbf{x})\,\mathrm{d}V^n = \int_{B^n} n\,\mathrm{d}V^n = n \cdot \operatorname{Vol}(B^n)
$$

3. Hypervolume of $n$-ball radius $R$: $\operatorname{Vol}(B^n) = \frac{\pi^{\frac{n}{2}}}{\Gamma(\frac{n}{2} + 1)} R^n$.
4. Outward flux $\Phi = \frac{n \pi^{\frac{n}{2}}}{\Gamma(\frac{n}{2} + 1)} R^n = \frac{2\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2}\right)} R^n = \text{Area}(\partial B^n) \cdot R$.

$$
\boxed{\Phi = n \cdot \operatorname{Vol}(B^n) = \frac{n\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2}+1\right)} R^n}
$$

#### Key Insight / Takeaway
Gauss's Divergence Theorem scales linearly with dimension $n$, equating radial position flux to $n$ times hypervolume.

---

The identity $n \cdot \operatorname{Vol}(B^n) = \operatorname{Area}(\partial B^n) \cdot R$ across dimensions, plus a Monte Carlo estimate of $\operatorname{Vol}(B^4)$.

In [22]:
# L3.4: radial flux through the n-sphere equals n * Vol(B^n), checked against the surface area.
from math import gamma
for n in range(2, 9):
    R = 1.7
    vol = np.pi ** (n / 2) / gamma(n / 2 + 1) * R**n
    flux_vol = n * vol
    area = 2 * np.pi ** (n / 2) / gamma(n / 2) * R ** (n - 1)
    flux_surf = area * R                       # F . n = R on the sphere
    print(f"n={n}: n*Vol = {flux_vol:.8f}   Area*R = {flux_surf:.8f}")
    assert rel(flux_vol, flux_surf) < 1e-12

# Monte Carlo check of Vol(B^n) for n = 4 against the closed form.
n, N = 4, 400_000
pts = rng.uniform(-1, 1, size=(N, n))
inside = (np.sum(pts**2, axis=1) <= 1).mean()
mc = inside * 2**n
print(f"\nMonte Carlo Vol(B^4) = {mc:.5f}   exact = {np.pi**2/2:.5f}")
assert abs(mc - np.pi**2 / 2) < 0.05

n=2: n*Vol = 18.15840554   Area*R = 18.15840554
n=3: n*Vol = 61.73857883   Area*R = 61.73857883
n=4: n*Vol = 164.86384584   Area*R = 164.86384584
n=5: n*Vol = 373.69138390   Area*R = 373.69138390
n=6: n*Vol = 748.41614280   Area*R = 748.41614280
n=7: n*Vol = 1357.12793895   Area*R = 1357.12793895
n=8: n*Vol = 2265.00730534   Area*R = 2265.00730534



Monte Carlo Vol(B^4) = 4.92528   exact = 4.93480


The two expressions agree to twelve digits in every dimension tested, and the Monte Carlo volume matches $\pi^2/2$ within sampling error.

---

### Problem L3.5 — Non-Simply Connected Topological Monodromy
**Source:** Spivak, *Calculus on Manifolds*, Ch. 5  

**Problem Statement:**  
Let $\mathbf{F}(x, y) = \frac{-y\mathbf{i} + x\mathbf{j}}{x^2 + y^2}$. Show that for any simple closed curve $C$ in $\mathbb{R}^2 \setminus \{(0,0)\}$, the line integral $\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 2\pi n$, where $n = \text{Winding}(C, (0,0))$ is the integer winding number of $C$ around the origin.

#### First-Principles Intuition
In polar coordinates, $\mathbf{F} \cdot \mathrm{d}\mathbf{r} = \mathrm{d}\theta$. Integrating $\mathrm{d}\theta$ around a loop measures the total change in polar angle $\Delta \theta = 2\pi n$.

#### Step-by-Step Solution
1. Express $\mathbf{F} \cdot \mathrm{d}\mathbf{r}$ in Cartesian terms:

$$
   \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \frac{-y\mathrm{d}x + x\mathrm{d}y}{x^2 + y^2}
$$

2. Recall polar relationship $\theta = \arctan(\frac{y}{x}) \implies \mathrm{d}\theta = \frac{x\mathrm{d}y - y\mathrm{d}x}{x^2 + y^2}$.
3. Thus $\mathbf{F} \cdot \mathrm{d}\mathbf{r} = \mathrm{d}\theta$ exact differential of angle.
4. Integrating $\mathrm{d}\theta$ along closed loop $C$:

$$
   \oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = \oint_C \mathrm{d}\theta = \Delta \theta
$$

5. Since $C$ is compact, $\theta$ can be lifted to a single continuous function $\tilde\theta: [0,1] \to \mathbb{R}$ along $\mathbf{r}(t)$, $t \in [0,1]$: the 1-form $\mathrm{d}\theta = \mathbf{F}\cdot\mathrm{d}\mathbf{r}$ is closed on $\mathbb{R}^2 \setminus \{(0,0)\}$, so a local antiderivative exists near every point of the loop, and these local branches patch into one continuous $\tilde\theta$ along $C$ by the standard path-lifting argument for a covering map ($\theta \mapsto e^{i\theta}$ covers $\mathbb{R}^2\setminus\{0\}$'s angular coordinate). Define $n := \frac{\tilde\theta(1) - \tilde\theta(0)}{2\pi}$; because $C$ is closed, $e^{i\tilde\theta(1)} = e^{i\tilde\theta(0)}$, forcing $n \in \mathbb{Z}$ — this integer is exactly the winding number $\text{Winding}(C,(0,0))$. Then $\oint_C \mathrm{d}\theta = \tilde\theta(1) - \tilde\theta(0) = 2\pi n$ by the single-variable FTC applied to $\tilde\theta$.

$$
\boxed{\oint_C \mathbf{F} \cdot \mathrm{d}\mathbf{r} = 2\pi \cdot \text{Winding}(C, (0,0))}
$$

For a **simple** closed curve $C$ (the hypothesis stated above), the Jordan Curve Theorem forces $\text{Winding}(C,(0,0)) \in \{0, \pm 1\}$ — $0$ if the origin lies outside $C$, $\pm 1$ if inside, with sign set by orientation; the general-$n$ phrasing only becomes meaningful once $C$ is allowed to self-intersect or wind repeatedly.

#### Key Insight / Takeaway
Line integrals of irrotational fields on non-simply connected domains measure topological invariants (winding numbers / de Rham cohomology).

---

Circulation around loops of several winding numbers, including one that avoids the origin.

In [23]:
# L3.5: circulation of the vortex field equals 2*pi times the winding number.
def circulation(loop, m=20000):
    """Trapezoid rule on a closed parameterised loop; the integrand is periodic."""
    t = np.linspace(0, 2 * np.pi, m, endpoint=False)
    x, y = loop(t)
    dt = 2 * np.pi / m
    dx = np.gradient(x, dt, edge_order=2)
    dy = np.gradient(y, dt, edge_order=2)
    d = x**2 + y**2
    return np.sum((-y * dx + x * dy) / d) * dt

for k in (-2, -1, 1, 2, 3):
    val = circulation(lambda t, k=k: ((2 + 0.3 * np.cos(3 * t)) * np.cos(k * t),
                                      (2 + 0.3 * np.cos(3 * t)) * np.sin(k * t)))
    print(f"winding {k:+d}: circulation = {val:.9f}   2*pi*n = {2*np.pi*k:.9f}")
    assert rel(val, 2 * np.pi * k) < 1e-6

# A loop that misses the origin: winding 0.
val0 = circulation(lambda t: (5 + np.cos(t), np.sin(t)))
print(f"loop avoiding the origin: circulation = {val0:.3e}")
assert abs(val0) < 1e-8

winding -2: circulation = -12.566369852   2*pi*n = -12.566370614
winding -1: circulation = -6.283185236   2*pi*n = -6.283185307
winding +1: circulation = 6.283185236   2*pi*n = 6.283185307
winding +2: circulation = 12.566369852   2*pi*n = 12.566370614
winding +3: circulation = 18.849553228   2*pi*n = 18.849555922
loop avoiding the origin: circulation = 5.168e-12


Circulation tracks $2\pi n$ to six digits for each winding number, and collapses to machine zero for the loop that avoids the puncture.

---

### Problem L3.6 — Vector Laplacian Identity & Electromagnetic Waves
**Source:** Feynman Lectures on Physics, Vol. II, Ch. 20  

**Problem Statement:**  
1. Derive vector Laplacian identity $\nabla \times (\nabla \times \mathbf{E}) = \nabla(\nabla \cdot \mathbf{E}) - \nabla^2 \mathbf{E}$.  
2. In free space ($\rho = 0, \mathbf{J} = \mathbf{0}$), use Maxwell's equations ($\nabla \cdot \mathbf{E} = 0, \nabla \times \mathbf{B} = \mu_0 \varepsilon_0 \frac{\partial \mathbf{E}}{\partial t}, \nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}$) to derive the 3D wave equation for electric field: $\nabla^2 \mathbf{E} - \mu_0 \varepsilon_0 \frac{\partial^2 \mathbf{E}}{\partial t^2} = \mathbf{0}$.

#### First-Principles Intuition
Taking the curl of Faraday's Law couples electric and magnetic field space-time partial derivatives into an uncoupled wave equation.

#### Step-by-Step Solution
1. Take curl of Faraday's Law $\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}$:

$$
   \nabla \times (\nabla \times \mathbf{E}) = \nabla \times \left( -\frac{\partial \mathbf{B}}{\partial t} \right) = -\frac{\partial}{\partial t}(\nabla \times \mathbf{B})
$$

2. Substitute Ampere-Maxwell Law in free space $\nabla \times \mathbf{B} = \mu_0 \varepsilon_0 \frac{\partial \mathbf{E}}{\partial t}$:

$$
   \nabla \times (\nabla \times \mathbf{E}) = -\frac{\partial}{\partial t}\left( \mu_0 \varepsilon_0 \frac{\partial \mathbf{E}}{\partial t} \right) = -\mu_0 \varepsilon_0 \frac{\partial^2 \mathbf{E}}{\partial t^2}
$$

3. Substitute vector Laplacian identity $\nabla \times (\nabla \times \mathbf{E}) = \nabla(\nabla \cdot \mathbf{E}) - \nabla^2 \mathbf{E}$:

$$
   \nabla(\nabla \cdot \mathbf{E}) - \nabla^2 \mathbf{E} = -\mu_0 \varepsilon_0 \frac{\partial^2 \mathbf{E}}{\partial t^2}
$$

4. In free space charge density is zero $\implies \nabla \cdot \mathbf{E} = 0$:

$$
   -\nabla^2 \mathbf{E} = -\mu_0 \varepsilon_0 \frac{\partial^2 \mathbf{E}}{\partial t^2} \implies \nabla^2 \mathbf{E} - \mu_0 \varepsilon_0 \frac{\partial^2 \mathbf{E}}{\partial t^2} = \mathbf{0}
$$

$$
\boxed{\nabla^2 \mathbf{E} - \frac{1}{c^2}\frac{\partial^2 \mathbf{E}}{\partial t^2} = \mathbf{0} \quad \text{where } c = \frac{1}{\sqrt{\mu_0 \varepsilon_0}}}
$$

#### Key Insight / Takeaway
Maxwell's field equations predict electromagnetic waves propagating at speed $c = \frac{1}{\sqrt{\mu_0 \varepsilon_0}}$ purely through vector calculus operator identities.

---

### Problem L3.7 — Surface Independence of Solenoidal Flux
**Source:** Pólya & Szegő, *Problems and Theorems in Analysis*, Vol. II  

**Problem Statement:**  
Let $\mathbf{F}$ be a solenoidal vector field ($\nabla \cdot \mathbf{F} = 0$) in $\mathbb{R}^3$. Prove that for any two oriented surfaces $S_1, S_2$ sharing the exact same boundary curve $\partial S_1 = \partial S_2 = C$, the flux through both surfaces is identical:

$$
\iint_{S_1} \mathbf{F} \cdot \mathbf{n}_1\,\mathrm{d}S = \iint_{S_2} \mathbf{F} \cdot \mathbf{n}_2\,\mathrm{d}S
$$

#### First-Principles Intuition
Joining $S_1$ and $-S_2$ forms a closed surface enclosing volume $V$. Apply Gauss's Divergence theorem to volume $V$.

#### Step-by-Step Solution
1. Construct closed surface $S = S_1 \cup (-S_2)$ with outward unit normal.
2. Enclosed volume $V$ has boundary $\partial V = S$.
3. Apply Gauss's Divergence Theorem:

$$
   \oiint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S = \iiint_V (\nabla \cdot \mathbf{F})\,\mathrm{d}V
$$

4. Since $\mathbf{F}$ is solenoidal, $\nabla \cdot \mathbf{F} = 0$:

$$
   \oiint_S \mathbf{F} \cdot \mathbf{n}\,\mathrm{d}S = \iiint_V 0\,\mathrm{d}V = 0
$$

5. Decompose closed surface flux:

$$
   \iint_{S_1} \mathbf{F} \cdot \mathbf{n}_1\,\mathrm{d}S + \iint_{-S_2} \mathbf{F} \cdot \mathbf{n}_{\text{in}}\,\mathrm{d}S = 0 \implies \iint_{S_1} \mathbf{F} \cdot \mathbf{n}_1\,\mathrm{d}S - \iint_{S_2} \mathbf{F} \cdot \mathbf{n}_2\,\mathrm{d}S = 0
$$

6. Therefore $\iint_{S_1} \mathbf{F} \cdot \mathbf{n}_1\,\mathrm{d}S = \iint_{S_2} \mathbf{F} \cdot \mathbf{n}_2\,\mathrm{d}S$.

$$
\boxed{\iint_{S_1} \mathbf{F} \cdot \mathbf{n}_1\,\mathrm{d}S = \iint_{S_2} \mathbf{F} \cdot \mathbf{n}_2\,\mathrm{d}S}
$$

#### Key Insight / Takeaway
Flux of a solenoidal field depends only on the bounding loop $C$, not on the shape of the surface spanning $C$.

---

### Problem L3.8 — Dirac Monopole Field & Non-Existent Global Vector Potential
**Source:** Spivak, *Calculus on Manifolds* / Mathematical Physics  

**Problem Statement:**  
Consider a magnetic monopole field $\mathbf{B}(\mathbf{r}) = g \frac{\hat{\mathbf{r}}}{r^2} = g \frac{\mathbf{r}}{r^3}$ on $U = \mathbb{R}^3 \setminus \{(0,0,0)\}$. Prove using Stokes' Theorem that no smooth global vector potential $\mathbf{A}$ satisfying $\nabla \times \mathbf{A} = \mathbf{B}$ can exist on $U$.

#### First-Principles Intuition
If a global vector potential existed, the total magnetic flux through the entire sphere $S^2$ would have to be zero by Stokes' Theorem. But the monopole produces total flux $4\pi g \neq 0$.

#### Step-by-Step Solution
1. Compute flux of monopole field through sphere $S^2$ of radius $R$:

$$
   \Phi = \oiint_{S^2} \mathbf{B} \cdot \mathbf{n}\,\mathrm{d}S = \oiint_{S^2} \frac{g}{R^2}\mathrm{d}S = \frac{g}{R^2}(4\pi R^2) = 4\pi g \neq 0
$$

2. Suppose for contradiction that a smooth global vector potential $\mathbf{A}$ exists on $U$ such that $\mathbf{B} = \nabla \times \mathbf{A}$.
3. Divide sphere $S^2$ into upper hemisphere $S_+$ and lower hemisphere $S_-$ sharing equator boundary $C$.
4. By Stokes' Theorem:

$$
   \iint_{S_+} (\nabla \times \mathbf{A}) \cdot \mathbf{n}\,\mathrm{d}S = \oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r}
$$

$$
   \iint_{S_-} (\nabla \times \mathbf{A}) \cdot \mathbf{n}\,\mathrm{d}S = \oint_{-C} \mathbf{A} \cdot \mathrm{d}\mathbf{r} = -\oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r}
$$

5. Total flux through sphere $S^2$:

$$
   \Phi = \oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r} - \oint_C \mathbf{A} \cdot \mathrm{d}\mathbf{r} = 0
$$

6. Contradiction ($4\pi g \neq 0$). Thus no single smooth global vector potential $\mathbf{A}$ exists!

$$
\boxed{\text{Contradiction: Global potential } \mathbf{A} \text{ cannot exist on } \mathbb{R}^3 \setminus \{\mathbf{0}\}}
$$

#### Key Insight / Takeaway
Magnetic monopoles require defining overlapping local vector potentials (Dirac strings / fiber bundles) rather than a single global vector potential.

---

### Problem L3.9 — Green's Second Identity and Dirichlet Uniqueness
**Source:** Cambridge University Mathematical Tripos, Part IB 2016  

**Problem Statement:**  
1. Prove Green's Second Identity:

$$
   \iiint_V (f\nabla^2 g - g\nabla^2 f)\,\mathrm{d}V = \oiint_{\partial V} (f\nabla g - g\nabla f) \cdot \mathbf{n}\,\mathrm{d}S
$$

2. Derive Green's First Identity $\iiint_V (u\nabla^2 u + \lVert\nabla u\rVert^2)\,\mathrm{d}V = \oiint_{\partial V} u\nabla u \cdot \mathbf{n}\,\mathrm{d}S$ (apply the divergence-product-rule argument of part 1 with $g=u$, $f=1$... more directly, apply Gauss's Divergence Theorem to $\mathbf{F}=u\nabla u$ and use $\nabla\cdot(u\nabla u)=\lVert\nabla u\rVert^2+u\nabla^2u$), then use it to prove that if $\nabla^2 u = 0$ on $V$ and $u = 0$ on boundary $\partial V$, then $u = 0$ everywhere inside $V$ (Dirichlet uniqueness).

#### First-Principles Intuition
Apply Divergence Theorem to vector fields $\mathbf{F} = f\nabla g$ and $\mathbf{G} = g\nabla f$ and subtract their divergence identities.

#### Step-by-Step Solution
1. Apply divergence vector product rule to $\mathbf{F} = f\nabla g$:

$$
   \nabla \cdot (f\nabla g) = \nabla f \cdot \nabla g + f \nabla^2 g
$$

2. Apply divergence vector product rule to $\mathbf{G} = g\nabla f$:

$$
   \nabla \cdot (g\nabla f) = \nabla g \cdot \nabla f + g \nabla^2 f
$$

3. Subtract both identities:

$$
   \nabla \cdot (f\nabla g - g\nabla f) = f\nabla^2 g - g\nabla^2 f
$$

4. Integrate over volume $V$ and apply Gauss's Divergence Theorem:

$$
   \iiint_V (f\nabla^2 g - g\nabla^2 f)\,\mathrm{d}V = \oiint_{\partial V} (f\nabla g - g\nabla f) \cdot \mathbf{n}\,\mathrm{d}S
$$

5. **Derive Green's First Identity.** Apply the divergence product rule to $\mathbf{F} = u\nabla u$: $\nabla \cdot (u\nabla u) = \nabla u \cdot \nabla u + u\nabla^2 u = \lVert\nabla u\rVert^2 + u\nabla^2 u$. Integrate over $V$ and apply Gauss's Divergence Theorem:

$$
   \iiint_V \big(u\nabla^2 u + \lVert\nabla u\rVert^2\big)\,\mathrm{d}V = \oiint_{\partial V} u\nabla u \cdot \mathbf{n}\,\mathrm{d}S
$$

   Now set $u$ with $\nabla^2 u = 0$ in $V$ and $u = 0$ on $\partial V$:

$$
   \iiint_V (0 + \lVert\nabla u\rVert^2)\,\mathrm{d}V = \oiint_{\partial V} 0 \cdot \nabla u \cdot \mathbf{n}\,\mathrm{d}S = 0 \implies \iiint_V \lVert\nabla u\rVert^2\,\mathrm{d}V = 0
$$

6. Since $\lVert\nabla u\rVert^2 \ge 0$, $\nabla u = \mathbf{0}$ everywhere in $V \implies u = \text{constant}$. Since $u=0$ on boundary, $u = 0$ everywhere in $V$.

$$
\boxed{\iiint_V \lVert\nabla u\rVert^2\,\mathrm{d}V = 0 \implies u = 0 \quad (\text{Unique Solution})}
$$

#### Key Insight / Takeaway
Green's identities establish uniqueness of boundary value solutions for partial differential equations (Poisson and Laplace equations).

---

### Problem L3.10 — Symplectic Decomposition and Game Stability
**Source:** Applied Mathematics & Machine Learning Research Synthesis  

**Problem Statement:**  
Consider a general linear 2D optimization velocity field:

$$
\mathbf{V}(x, y) = \begin{pmatrix} a & b \\ c & d \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = (ax+by)\mathbf{i} + (cx+dy)\mathbf{j}
$$

1. Compute $\nabla \cdot \mathbf{V}$ and $\nabla \times \mathbf{F}$ (where $\mathbf{F} = (ax+by, cx+dy, 0)$).  
2. Decompose $\mathbf{V}$ into gradient flow field $\mathbf{V}_{\operatorname{grad}} = -\nabla \Phi$ and Hamiltonian rotational flow field $\mathbf{V}_{\text{rot}} = \mathbf{J} \nabla H$ where:

$$
   \mathbf{J} = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}
$$

3. Prove that phase space trajectories spiral inward to origin $(0,0)$ if and only if divergence is strictly negative ($\operatorname{Tr}(A) = a+d \lt 0$) and $\det(A) = ad - bc \gt 0$.

#### First-Principles Intuition
Vector calculus operators isolate energy-dissipative gradient dynamics ($\nabla \cdot \mathbf{V} = \operatorname{Tr}(A)$) from energy-preserving Hamiltonian rotational dynamics ($\nabla \times \mathbf{V} = c-b$).

#### Step-by-Step Solution
1. Compute field operators:
   - Divergence: $\nabla \cdot \mathbf{V} = \frac{\partial}{\partial x}(ax+by) + \frac{\partial}{\partial y}(cx+dy) = a + d = \operatorname{Tr}(A)$.
   - Curl: $\nabla \times \mathbf{F} = \left(\frac{\partial}{\partial x}(cx+dy) - \frac{\partial}{\partial y}(ax+by)\right)\mathbf{k} = (c - b)\mathbf{k}$.
2. Decompose matrix $A$:

$$
   A = S + K = \begin{pmatrix} a & \frac{b+c}{2} \\ \frac{b+c}{2} & d \end{pmatrix} + \begin{pmatrix} 0 & \frac{b-c}{2} \\ -\frac{b-c}{2} & 0 \end{pmatrix}
$$

   - Symmetric part $S$ generates gradient potential $\Phi(x, y) = -\frac{1}{2}(ax^2 + (b+c)xy + dy^2)$, so $\mathbf{V}_{\operatorname{grad}} = -\nabla \Phi = S \mathbf{x}$.
   - Skew-symmetric part $K$ generates Hamiltonian $H(x, y) = \frac{b-c}{4}(x^2 + y^2)$: with $\mathbf{J} = \begin{pmatrix}0&1\\-1&0\end{pmatrix}$, $\nabla H = \frac{b-c}{2}(x, y)$, so $\mathbf{J}\nabla H = \frac{b-c}{2}(y, -x) = K\mathbf{x}$, giving $\mathbf{V}_{\text{rot}} = \mathbf{J}\nabla H = K \mathbf{x}$. (Note the sign: $H = \frac{c-b}{4}(x^2+y^2)$ would give $\mathbf{J}\nabla H = -K\mathbf{x}$, the wrong sign.)
3. Linear dynamical system stability analysis:
   Eigenvalues of matrix $A$ satisfy characteristic polynomial $\lambda^2 - \operatorname{Tr}(A)\lambda + \det(A) = 0$:

$$
   \lambda_{1,2} = \frac{\operatorname{Tr}(A) \pm \sqrt{\operatorname{Tr}(A)^2 - 4\det(A)}}{2}
$$

   Trajectories tend to the origin exactly when both eigenvalues have strictly negative real part.
   For a real $2 \times 2$ matrix, $\lambda_1 + \lambda_2 = \operatorname{Tr}(A)$ and
   $\lambda_1\lambda_2 = \det(A)$, so both real parts are negative if and only if
   $\operatorname{Tr}(A) \lt 0$ **and** $\det(A) \gt 0$ (the Routh-Hurwitz criterion in dimension two):
   - if $\det(A) \le 0$ the eigenvalues are real of opposite sign, or one is zero, so the origin is a saddle or is not attracting;
   - if $\det(A) \gt 0$ the eigenvalues have the same sign of real part, and its sign is that of $\operatorname{Tr}(A)$.

   The approach is a **spiral** (complex eigenvalues) precisely when in addition
   $\operatorname{Tr}(A)^2 \lt 4\det(A)$; when $\operatorname{Tr}(A)^2 \ge 4\det(A)$ the eigenvalues
   are real and the origin is a stable node, approached without rotation. Curl controls which:
   $\operatorname{Tr}(A)^2 - 4\det(A) = (a-d)^2 + (b+c)^2 - (c-b)^2$, so a large curl $c - b$ is what
   pushes the discriminant negative.

$$
\boxed{\nabla \cdot \mathbf{V} = a+d, \quad \nabla \times \mathbf{F} = (c-b)\mathbf{k}, \quad \text{origin attracting} \iff \operatorname{Tr}(A) \lt 0 \text{ and } \det(A) \gt 0, \quad \text{spiral} \iff \text{additionally } \operatorname{Tr}(A)^2 \lt 4\det(A)}
$$

#### Key Insight / Takeaway
The divergence $\nabla \cdot \mathbf{V} = \operatorname{Tr}(A)$ measures the phase-space volume contraction rate and decides *whether* trajectories converge; the curl $c - b$ decides *how* they converge, spiralling rather than approaching along a straight line.

The stability and spiral criteria against 4000 random matrices, comparing the predicted classification with the eigenvalues NumPy actually returns.

In [24]:
# L3.10: Routh-Hurwitz in the plane -- Tr < 0 and det > 0 is exactly asymptotic stability,
# and Tr^2 < 4 det is exactly the spiral case.
A = rng.normal(size=(4000, 2, 2))
tr = A[:, 0, 0] + A[:, 1, 1]
det = A[:, 0, 0] * A[:, 1, 1] - A[:, 0, 1] * A[:, 1, 0]
eig = np.linalg.eigvals(A)
stable = eig.real.max(axis=1) < 0
predicted = (tr < 0) & (det > 0)
print("stability criterion mismatches:", int((stable != predicted).sum()), "of", len(A))

spiral = np.abs(eig.imag).max(axis=1) > 1e-12
print("spiral criterion mismatches:   ", int((spiral != (tr**2 < 4 * det)).sum()), "of", len(A))

# The discriminant identity that ties the spiral case to the curl.
curlv = A[:, 1, 0] - A[:, 0, 1]
disc_id = (A[:, 0, 0] - A[:, 1, 1])**2 + (A[:, 0, 1] + A[:, 1, 0])**2 - curlv**2
print("max |Tr^2 - 4det - [(a-d)^2 + (b+c)^2 - (c-b)^2]| =",
      np.abs(tr**2 - 4 * det - disc_id).max())
assert np.abs(tr**2 - 4 * det - disc_id).max() < 1e-10
assert (stable == predicted).all()
assert (spiral == (tr**2 < 4 * det)).all()

stability criterion mismatches: 0 of 4000
spiral criterion mismatches:    0 of 4000
max |Tr^2 - 4det - [(a-d)^2 + (b+c)^2 - (c-b)^2]| = 7.105427357601002e-15


Zero mismatches on both criteria across 4000 random matrices: $\operatorname{Tr}(A) \lt 0$ with $\det(A) \gt 0$ is precisely asymptotic stability, and $\operatorname{Tr}(A)^2 \lt 4\det(A)$ is precisely the spiral case.

---